In [1]:
"""
=============================================================================
  SPAN v4 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  CME Standard Portfolio Analysis of Risk (SPAN), Version 4
  16-Scenario Grid  |  PSR · ICSC · VSR · NOV · SOM
=============================================================================
  Portfolio  : Loaded from Common_Portfolio_SPAN_SPAN2_STANS.xlsx
  Exchange   : MCX (Multi Commodity Exchange of India)
  Date       : June 2026
=============================================================================
"""

import os, sys, math
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD PORTFOLIO FROM EXCEL
# ─────────────────────────────────────────────────────────────────────────────

def load_portfolio(xlsx_path: str) -> list:
    """
    Read the 'Common_Portfolio' sheet from the workbook.
    Column mapping (0-indexed):
      0  Row#          1  Commodity     2  CC Group
      3  Inst Type     4  Buy/Sell      5  Qty(Lots)
      6  Contract Month 7 Futures Price 8  Option Premium
      9  Strike Price  10 Option Type  11 Implied Vol(%)
      12 Lot Size(MT)  13 Contract Val 14 Approx Delta
      15 Position Tag  16 SPAN Component 17 Framework Relevance
    """
    import openpyxl
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb["Common_Portfolio"]

    portfolio = []
    for row in ws.iter_rows(min_row=3, values_only=True):   # row 1=title, row 2=headers
        row_num = row[0]
        if row_num is None or not isinstance(row_num, (int, float)):
            break   # stop at colour-legend section

        commodity = str(row[1]).strip()
        cc        = str(row[2]).strip()
        inst      = str(row[3]).strip()    # "Future" or "Option"
        side      = str(row[4]).strip()    # "Buy" or "Sell"
        qty       = int(row[5])
        month     = str(row[6]).strip()
        F         = float(row[7])  if row[7]  is not None else None   # futures price
        P         = float(row[8])  if row[8]  is not None else None   # option premium
        K         = float(row[9])  if row[9]  is not None else None   # strike
        cp        = str(row[10]).strip() if row[10] is not None else None  # Call/Put
        iv_raw    = row[11]
        iv        = float(iv_raw) / 100 if iv_raw is not None else None  # convert % → decimal
        lot       = float(row[12])
        delta     = float(row[14]) if row[14] is not None else None

        # For options, underlying futures price must be populated from col 7
        # (workbook stores it there for options too via context)
        if inst == "Option" and F is None:
            # fall back: try to infer from commodity reference prices
            ref = {"Jeera": 22285, "Guar Gum": 10756, "Castor Seed": 6392,
                   "Kapas": 1682, "Guar Seed": 5715, "Coriander": 13350}
            F = ref.get(commodity, 0)

        portfolio.append(dict(
            row       = int(row_num),
            commodity = commodity,
            cc        = cc,
            inst      = inst,
            side      = side,
            qty       = qty,
            month     = month,
            F         = F,
            P         = P,
            K         = K,
            cp        = cp,
            iv        = iv,
            lot       = lot,
            delta     = delta,
        ))

    print(f"  [Loaded {len(portfolio)} positions from '{xlsx_path}']")
    return portfolio


# ── Locate the workbook – searches common locations automatically ─────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"

_SEARCH_DIRS = [
    "/content",                                              # Google Colab default
    os.getcwd(),                                             # current working dir
    "/sessions/busy-intelligent-johnson/mnt/uploads",        # Cowork sandbox
]
# also try the script's own directory if __file__ is available
try:
    _SEARCH_DIRS.append(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass

xlsx_path = None
for _d in _SEARCH_DIRS:
    _candidate = os.path.join(_d, _FILENAME)
    if os.path.exists(_candidate):
        xlsx_path = _candidate
        break

if xlsx_path is None:
    sys.exit(f"ERROR: Could not find {_FILENAME}.\n"
             f"Upload it to Colab (/content/) or place it in the same folder as this script.")

PORTFOLIO = load_portfolio(xlsx_path)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  SPAN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
# Price-scan range as % of futures price (MCX approximation)
PSR_PCT = {
    "Turmeric":         0.05,   # 5%
    "Coriander":        0.05,
    "Jeera":            0.05,
    "Guar Gum":         0.05,
    "Guar Seed":        0.05,
    "Castor Seed":      0.05,
    "Cotton Seed OC":   0.05,
    "Kapas":            0.05,
    "Kapas (Cotton)":   0.05,   # exact name from Excel
}
# Vol-scan range (± fraction of implied vol) used in 16-scenario array
VOL_SHIFT_FRAC = 0.25          # ±25% of IV

# Intra-CC Spread Charge (ICSC) per spread pair in ₹/lot
ICSC_CHARGE = {
    "Turmeric":  8050,    # per spread (2 legs = 1 spread)
    "Guar Seed": 5750,
}

# Inter-CC Spread Credit (% of combined PSR) – SPAN fixed table
INTER_CC_CREDIT = {
    ("Guar Gum", "Guar Seed"):                0.50,
    ("Cotton Seed OC", "Kapas"):              0.40,
    ("Cotton Seed OC", "Kapas (Cotton)"):     0.40,
}

# SOM — also add Kapas (Cotton) alias
SOM_PCT_EXTRA = {"Kapas (Cotton)": 0.005}

# Short Option Minimum rate (% of futures price × lot size × qty × 100)
SOM_PCT = {
    "Turmeric":       0.005,
    "Coriander":      0.005,
    "Jeera":          0.005,
    "Guar Gum":       0.005,
    "Guar Seed":      0.005,
    "Castor Seed":    0.005,
    "Cotton Seed OC":  0.005,
    "Kapas":           0.005,
    "Kapas (Cotton)":  0.005,
}

T_YEARS = 1/12    # ~1 month to expiry (Apr-26 options)

# ─────────────────────────────────────────────────────────────────────────────
# 2.  BLACK-SCHOLES OPTION PRICER
# ─────────────────────────────────────────────────────────────────────────────
def bs_price(F, K, T, sigma, cp):
    """Black-Scholes for futures option (r=0 convenience)."""
    if T <= 0 or sigma <= 0 or F <= 0 or K <= 0:
        intrinsic = max(0, F - K) if cp == "Call" else max(0, K - F)
        return intrinsic
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    from scipy.stats import norm
    if cp == "Call":
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    else:
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

def bs_delta(F, K, T, sigma, cp):
    if T <= 0:
        return 1.0 if (cp == "Call" and F > K) else (-1.0 if (cp == "Put" and F < K) else 0.0)
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    from scipy.stats import norm
    return norm.cdf(d1) if cp == "Call" else norm.cdf(d1) - 1

# ─────────────────────────────────────────────────────────────────────────────
# 3.  SPAN 16-SCENARIO ARRAY DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
def build_scenario_array():
    """
    Classic SPAN 16-scenario grid:
    Scenarios 1-14 : price moves ±3×, ±2×, ±1×, ±⅓× PSR  ×  vol up/dn
    Scenario 15-16 : extreme price moves (3× PSR) with fractional (33%) weight
    Returns list of (price_frac, vol_shift, weight) tuples.
    price_frac  = multiplier on PSR (e.g. +1.0 means F × (1 + PSR_PCT))
    vol_shift   = additive fraction to IV (e.g. +0.25 means IV × 1.25)
    weight      = scenario weight (1.0 full, 0.333 fractional)
    """
    price_moves   = [+3, +2, +1, +0.333, -0.333, -1, -2, -3]
    vol_moves     = [+VOL_SHIFT_FRAC, -VOL_SHIFT_FRAC]
    scenarios = []
    for pm in price_moves:
        for vm in vol_moves:
            weight = 0.333 if abs(pm) == 3 else 1.0
            scenarios.append((pm, vm, weight))    # 16 total
    return scenarios   # length = 16

SCENARIOS = build_scenario_array()

# ─────────────────────────────────────────────────────────────────────────────
# 4.  PER-POSITION SCAN RISK (PSR / VSR)
# ─────────────────────────────────────────────────────────────────────────────
def compute_scan_risk(pos):
    """
    For each position, compute P&L across all 16 scenarios.
    Worst-case loss = Scan Risk (PSR for futures, VSR for options).
    Returns:
        scan_risk      : worst-case loss (positive = margin required)
        scan_array     : list of 16 (scenario_label, P&L) tuples
        nov            : net option value (premium × qty × lot × 100)
                         negative if long (we pay premium), positive if short
    """
    F        = pos["F"]
    lot      = pos["lot"]
    qty      = pos["qty"]
    side_sgn = 1 if pos["side"] == "Buy" else -1
    multiplier = qty * lot * 100    # lot × 100 qtl/MT standard (MCX)

    scan_results = []

    if pos["inst"] == "Future":
        psr_range = F * PSR_PCT[pos["commodity"]]
        for i, (pm, vm, wt) in enumerate(SCENARIOS, 1):
            F_new  = F + pm * psr_range
            pnl    = side_sgn * (F_new - F) * multiplier
            scan_results.append({
                "scen": i,
                "price_move": f"{'+' if pm>=0 else ''}{pm:.2f}×PSR",
                "vol_move":   f"{'+' if vm>=0 else ''}{vm*100:.0f}%vol",
                "F_new": round(F_new, 2),
                "pnl": round(pnl, 0),
                "weighted_loss": round(-pnl * wt, 0)
            })
        nov = 0.0
    else:
        # Option
        iv     = pos["iv"]
        K      = pos["K"]
        cp     = pos["cp"]
        psr_range = F * PSR_PCT[pos["commodity"]]
        # Current theoretical price
        P0     = bs_price(F, K, T_YEARS, iv, cp)
        for i, (pm, vm, wt) in enumerate(SCENARIOS, 1):
            F_new  = F + pm * psr_range
            iv_new = max(0.001, iv * (1 + vm))
            P_new  = bs_price(F_new, K, T_YEARS, iv_new, cp)
            pnl    = side_sgn * (P_new - P0) * multiplier
            scan_results.append({
                "scen": i,
                "price_move": f"{'+' if pm>=0 else ''}{pm:.2f}×PSR",
                "vol_move":   f"{'+' if vm>=0 else ''}{vm*100:.0f}%vol",
                "F_new": round(F_new, 2),
                "iv_new": round(iv_new, 4),
                "P_new": round(P_new, 2),
                "pnl": round(pnl, 0),
                "weighted_loss": round(-pnl * wt, 0)
            })
        # NOV = current market value from position's perspective
        nov = side_sgn * P0 * multiplier   # positive = credit if short + ITM

    # Scan risk = worst weighted loss across all scenarios
    worst_idx = max(range(len(scan_results)),
                    key=lambda i: scan_results[i]["weighted_loss"])
    scan_risk = max(0, scan_results[worst_idx]["weighted_loss"])

    return scan_risk, scan_results, nov

# ─────────────────────────────────────────────────────────────────────────────
# 5.  SHORT OPTION MINIMUM (SOM)
# ─────────────────────────────────────────────────────────────────────────────
def compute_som(pos):
    """
    SOM = SOM_PCT × F × lot × 100 × qty
    Applied only to short option positions.
    Final margin for a short option = max(scan_risk - |NOV|, SOM)
    """
    if pos["inst"] != "Option" or pos["side"] != "Sell":
        return 0.0
    F   = pos["F"]
    qty = pos["qty"]
    lot = pos["lot"]
    return SOM_PCT[pos["commodity"]] * F * lot * 100 * qty

# ─────────────────────────────────────────────────────────────────────────────
# 6.  INTRA-CC SPREAD CHARGE (ICSC)
# ─────────────────────────────────────────────────────────────────────────────
def identify_calendar_spreads(portfolio):
    """
    Detect matched calendar spread pairs: same commodity, same CC,
    both Future, opposite sides, different months.
    Returns list of (buy_row, sell_row, spread_qty, commodity) tuples.
    """
    spreads = []
    futures = [p for p in portfolio if p["inst"] == "Future"]
    processed = set()
    for i, p1 in enumerate(futures):
        if i in processed:
            continue
        for j, p2 in enumerate(futures):
            if j <= i or j in processed:
                continue
            if (p1["commodity"] == p2["commodity"] and
                p1["cc"] == p2["cc"] and
                p1["month"] != p2["month"] and
                p1["side"] != p2["side"]):
                spread_qty = min(p1["qty"], p2["qty"])
                spreads.append({
                    "commodity": p1["commodity"],
                    "cc": p1["cc"],
                    "near_row": p1["row"],
                    "far_row": p2["row"],
                    "spread_qty": spread_qty
                })
                processed.add(i)
                processed.add(j)
    return spreads

# ─────────────────────────────────────────────────────────────────────────────
# 7.  INTER-CC SPREAD CREDIT
# ─────────────────────────────────────────────────────────────────────────────
def compute_inter_cc_credits(portfolio, psr_by_row):
    """
    For paired commodities (e.g. Guar Gum ↔ Guar Seed),
    grant a credit equal to INTER_CC_CREDIT% of the smaller position's PSR.
    """
    credits = {}

    def get_net_psr(commodity):
        rows = [p["row"] for p in portfolio if p["commodity"] == commodity and p["inst"] == "Future"]
        return sum(psr_by_row.get(r, 0) for r in rows)

    for (c1, c2), rate in INTER_CC_CREDIT.items():
        psr1 = get_net_psr(c1)
        psr2 = get_net_psr(c2)
        credit = rate * min(psr1, psr2)
        credits[(c1, c2)] = credit

    return credits

# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN SPAN CALCULATION
# ─────────────────────────────────────────────────────────────────────────────
def run_span(portfolio, verbose=True):
    sep  = "=" * 110
    sep2 = "-" * 110

    print(sep)
    print("  SPAN v4 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio")
    print("  Standard Portfolio Analysis of Risk (CME SPAN v4)  |  16-Scenario Grid")
    print(sep)

    # ── Step 1: Per-position scan risk ────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 1 : PER-POSITION SCAN RISK  (16-Scenario P&L Array)")
    print(sep2)

    results = []
    psr_by_row = {}

    for pos in portfolio:
        scan_risk, scan_arr, nov = compute_scan_risk(pos)
        som                       = compute_som(pos)
        psr_by_row[pos["row"]]    = scan_risk
        results.append({
            "pos":       pos,
            "scan_risk": scan_risk,
            "scan_arr":  scan_arr,
            "nov":       nov,
            "som":       som,
        })

        # ── Detailed per-position output ──────────────────────────────────────
        sign  = "+" if pos["side"] == "Buy" else "-"
        label = f"Row {pos['row']:2d} | {pos['commodity']:15s} | {pos['inst']:6s} | {pos['side']:4s} × {pos['qty']} lot"
        print(f"\n  {'─'*80}")
        print(f"  {label}")
        if pos["inst"] == "Future":
            cv = pos["F"] * pos["lot"] * 100 * pos["qty"]
            print(f"  Futures Price : ₹{pos['F']:>10,.2f}/Qtl   "
                  f"Lot Size : {pos['lot']} MT   "
                  f"Contract Value : ₹{cv:>12,.0f}")
            print(f"  PSR Range     : ±₹{pos['F']*PSR_PCT[pos['commodity']]*pos['lot']*100*pos['qty']:>10,.0f}  "
                  f"({PSR_PCT[pos['commodity']]*100:.1f}% × contract value)")
        else:
            cv = pos["P"] * pos["lot"] * 100 * pos["qty"]
            print(f"  Underlying    : ₹{pos['F']:>10,.2f}   "
                  f"Strike : ₹{pos['K']:,.2f}  {pos['cp']:4s}  "
                  f"IV : {pos['iv']*100:.1f}%   "
                  f"Δ : {pos['delta']:+.2f}")
            print(f"  Option B-S Px : ₹{bs_price(pos['F'],pos['K'],T_YEARS,pos['iv'],pos['cp']):>8,.2f}   "
                  f"Premium Paid/Rcvd : ₹{cv:>10,.0f}   "
                  f"NOV : ₹{nov:>10,.0f}")

        # Print the 16 scenario table
        tbl_rows = []
        for s in scan_arr:
            marker = " ◄ WORST" if s["weighted_loss"] == scan_risk and scan_risk > 0 else ""
            if pos["inst"] == "Future":
                tbl_rows.append([
                    f"S{s['scen']:02d}",
                    s["price_move"],
                    s["vol_move"],
                    f"₹{s['F_new']:>10,.2f}",
                    "—",
                    f"₹{s['pnl']:>12,.0f}",
                    f"₹{s['weighted_loss']:>12,.0f}" + marker
                ])
            else:
                tbl_rows.append([
                    f"S{s['scen']:02d}",
                    s["price_move"],
                    s["vol_move"],
                    f"₹{s['F_new']:>10,.2f}",
                    f"₹{s['P_new']:>8,.2f}",
                    f"₹{s['pnl']:>12,.0f}",
                    f"₹{s['weighted_loss']:>12,.0f}" + marker
                ])

        print()
        print(tabulate(tbl_rows,
                       headers=["Scen", "Price Move", "Vol Move",
                                 "F_new / Undlg", "Opt Px New",
                                 "Position P&L (₹)", "Wtd Loss (₹)"],
                       tablefmt="simple", colalign=("left","left","left","right","right","right","right")))

        print(f"\n  ► Scan Risk (worst weighted loss)  : ₹{scan_risk:>12,.0f}")
        if pos["inst"] == "Option":
            if pos["side"] == "Sell":
                print(f"  ► Short Option Minimum (SOM)       : ₹{som:>12,.0f}")
            print(f"  ► Net Option Value  (NOV credit)   : ₹{nov:>12,.0f}")

    # ── Step 2: Calendar Spread ICSC ─────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 2 : INTRA-CC SPREAD CHARGE (ICSC)  –  Calendar Spread Pairs")
    print(sep2)

    cal_spreads = identify_calendar_spreads(portfolio)
    total_icsc  = 0

    if not cal_spreads:
        print("  No calendar spread pairs detected.")
    else:
        icsc_tbl = []
        for sp in cal_spreads:
            charge_per = ICSC_CHARGE.get(sp["commodity"], 0)
            charge_tot = charge_per * sp["spread_qty"]
            total_icsc += charge_tot * 2   # both legs charged
            icsc_tbl.append([
                sp["commodity"],
                sp["cc"],
                f"Row {sp['near_row']}",
                f"Row {sp['far_row']}",
                sp["spread_qty"],
                f"₹{charge_per:>8,.0f}",
                f"₹{charge_tot*2:>10,.0f}  (both legs)"
            ])
            print(f"\n  {sp['commodity']} ({sp['cc']}) : Row {sp['near_row']} (near) ↔ Row {sp['far_row']} (far)")
            print(f"    Spread Qty : {sp['spread_qty']} lot(s)")
            print(f"    ICSC Rate  : ₹{charge_per:,.0f} per lot pair")
            print(f"    ICSC Total : ₹{charge_tot*2:,.0f} (charged symmetrically to both legs)")

        print()
        print(tabulate(icsc_tbl,
                       headers=["Commodity","CC","Near Leg","Far Leg","Qty","Rate/Lot","Total ICSC (₹)"],
                       tablefmt="simple"))
        print(f"\n  Total ICSC Charge  :  ₹{total_icsc:>12,.0f}")

    # ── Step 3: Inter-CC Spread Credits ──────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 3 : INTER-CC SPREAD CREDIT  –  Correlated Commodity Pairs")
    print(sep2)

    inter_credits = compute_inter_cc_credits(portfolio, psr_by_row)
    total_inter_cc_credit = 0

    cc_credit_tbl = []
    for (c1, c2), credit in inter_credits.items():
        rate = INTER_CC_CREDIT[(c1, c2)]
        psr1 = sum(psr_by_row.get(p["row"], 0)
                   for p in portfolio if p["commodity"] == c1 and p["inst"] == "Future")
        psr2 = sum(psr_by_row.get(p["row"], 0)
                   for p in portfolio if p["commodity"] == c2 and p["inst"] == "Future")
        total_inter_cc_credit += credit
        cc_credit_tbl.append([
            f"{c1} ↔ {c2}",
            f"₹{psr1:>10,.0f}",
            f"₹{psr2:>10,.0f}",
            f"{rate*100:.0f}%",
            f"₹{credit:>10,.0f}"
        ])
        print(f"\n  Pair : {c1} ↔ {c2}")
        print(f"    {c1} PSR Total : ₹{psr1:,.0f}")
        print(f"    {c2} PSR Total : ₹{psr2:,.0f}")
        print(f"    Credit Rate    : {rate*100:.0f}% of min(PSR1, PSR2)")
        print(f"    Credit Amount  : ₹{credit:,.0f}")

    print()
    print(tabulate(cc_credit_tbl,
                   headers=["CC Pair", "PSR (C1)", "PSR (C2)", "Credit Rate", "Credit (₹)"],
                   tablefmt="simple"))
    print(f"\n  Total Inter-CC Credit  :  ₹{total_inter_cc_credit:>12,.0f}")

    # ── Step 4: NOV Offsets & SOM Floors ─────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 4 : NET OPTION VALUE (NOV) CREDIT  &  SHORT OPTION MINIMUM (SOM)")
    print(sep2)

    nov_tbl = []
    for r in results:
        pos = r["pos"]
        if pos["inst"] != "Option":
            continue
        scan  = r["scan_risk"]
        nov   = r["nov"]
        som   = r["som"]

        if pos["side"] == "Buy":
            # Long option: scan risk reduced by NOV (capped at 0)
            final = max(0, scan + nov)   # nov is negative for long (cost)
            component = "Scan Risk reduced by premium cost"
        else:
            # Short option: max(Scan - NOV_credit, SOM)
            nov_credit = abs(nov)        # premium received
            raw = max(0, scan - nov_credit)
            final = max(raw, som)
            component = f"max(Scan - NOV_credit={nov_credit:,.0f}, SOM={som:,.0f})"

        nov_tbl.append([
            f"Row {pos['row']:2d}",
            pos["commodity"],
            pos["side"],
            f"₹{scan:>10,.0f}",
            f"₹{nov:>10,.0f}",
            f"₹{som:>10,.0f}",
            f"₹{final:>10,.0f}",
        ])
        r["final_option_margin"] = final

    print()
    print(tabulate(nov_tbl,
                   headers=["Row","Commodity","Side","Scan Risk","NOV","SOM Floor","Final Margin"],
                   tablefmt="simple",
                   colalign=("left","left","left","right","right","right","right")))

    # ── Step 5: Aggregate Per-Position Margin ─────────────────────────────────
    print("\n" + sep2)
    print("  STEP 5 : POSITION-LEVEL MARGIN SUMMARY")
    print(sep2)

    summary_tbl   = []
    gross_total   = 0
    span_component_totals = {}

    for r in results:
        pos  = r["pos"]
        scan = r["scan_risk"]
        nov  = r["nov"]
        som  = r["som"]

        if pos["inst"] == "Future":
            margin = scan
            comp   = pos.get("span_comp", "PSR")
        else:
            margin = r.get("final_option_margin",
                           max(0, scan + nov) if pos["side"] == "Buy"
                           else max(0, scan - abs(nov), som))
            comp = "VSR+NOV" if pos["side"] == "Buy" else ("SOM" if som >= scan - abs(nov) else "VSR+NOV+SOM")

        gross_total += margin
        tag = pos.get("span_comp", comp)
        span_component_totals[tag] = span_component_totals.get(tag, 0) + margin

        cv = (pos["F"] * pos["lot"] * 100 * pos["qty"]) if pos["inst"] == "Future" \
             else (pos["P"] * pos["lot"] * 100 * pos["qty"])

        summary_tbl.append([
            pos["row"],
            pos["commodity"],
            pos["inst"],
            pos["side"],
            pos["qty"],
            f"₹{cv:>12,.0f}",
            f"₹{scan:>12,.0f}",
            f"₹{margin:>12,.0f}",
        ])

    print()
    print(tabulate(summary_tbl,
                   headers=["Row","Commodity","Type","Side","Qty",
                             "Contract Value","Scan Risk (₹)","Margin Req (₹)"],
                   tablefmt="simple",
                   colalign=("right","left","left","left","right","right","right","right")))

    # ── Step 6: Deduct Credits ────────────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 6 : APPLY SPREAD CREDITS  →  NET SPAN MARGIN")
    print(sep2)

    net_margin = gross_total - total_icsc - total_inter_cc_credit

    print(f"\n  Gross Scan Risk (sum of all position margins)  : ₹{gross_total:>15,.0f}")
    print(f"  Less : Intra-CC Spread Charges (ICSC)          : ₹{total_icsc:>15,.0f}  (already embedded in gross)")
    print(f"  Less : Inter-CC Spread Credits                 : ₹{total_inter_cc_credit:>15,.0f}")
    print(f"  {'─'*60}")
    print(f"  NET SPAN INITIAL MARGIN                        : ₹{net_margin:>15,.0f}")

    # ── Step 7: Component Breakdown ───────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 7 : MARGIN BY SPAN COMPONENT")
    print(sep2)

    comp_map = {}
    for r in results:
        pos = r["pos"]
        if pos["inst"] == "Future":
            if pos["row"] in [9, 10, 11, 12]:
                key = "ICSC (Calendar Spread)"
            else:
                key = "PSR (Price Scan Risk)"
        else:
            som = r["som"]
            scan = r["scan_risk"]
            nov = abs(r["nov"])
            final_m = r.get("final_option_margin", max(0, scan - nov, som) if pos["side"] == "Sell" else max(0, scan - nov))
            if pos["side"] == "Sell" and som >= max(0, scan - nov):
                key = "SOM (Short Option Minimum)"
            else:
                key = "VSR + NOV (Option Scan)"
            r["final_option_margin"] = final_m

        margin = r.get("final_option_margin", r["scan_risk"]) if pos["inst"] == "Option" else r["scan_risk"]
        comp_map[key] = comp_map.get(key, 0) + margin

    comp_tbl = [[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"] for k, v in comp_map.items()]
    comp_tbl.append(["TOTAL (Gross)", f"₹{gross_total:>12,.0f}", "100.0%"])
    print()
    print(tabulate(comp_tbl,
                   headers=["SPAN Component","Margin (₹)","% of Gross"],
                   tablefmt="simple",
                   colalign=("left","right","right")))

    # ── Step 8: CC-Group Breakdown ────────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 8 : MARGIN BY COMMODITY GROUP (CC)")
    print(sep2)

    cc_map = {}
    for r in results:
        pos    = r["pos"]
        margin = r.get("final_option_margin", r["scan_risk"]) if pos["inst"] == "Option" else r["scan_risk"]
        cc_map[pos["cc"]] = cc_map.get(pos["cc"], 0) + margin

    cc_tbl = sorted([[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"] for k, v in cc_map.items()],
                    key=lambda x: -float(x[1].replace("₹","").replace(",","")))
    print()
    print(tabulate(cc_tbl,
                   headers=["CC Group","Margin (₹)","% of Total"],
                   tablefmt="simple",
                   colalign=("left","right","right")))

    # ── Final Summary Banner ───────────────────────────────────────────────────
    print("\n" + sep)
    print("  SPAN v4  FINAL MARGIN SUMMARY")
    print(sep)
    print(f"  Total Positions          : {len(portfolio)}")
    print(f"  Futures Positions        : {sum(1 for p in portfolio if p['inst']=='Future')}")
    print(f"  Options Positions        : {sum(1 for p in portfolio if p['inst']=='Option')}")
    print(f"  Calendar Spread Pairs    : {len(cal_spreads)}")
    print(f"  Inter-CC Spread Pairs    : {len(inter_credits)}")
    print()
    print(f"  Gross Scan Risk          : ₹{gross_total:>15,.0f}")
    print(f"  Inter-CC Credits         : ₹{total_inter_cc_credit:>15,.0f}")
    print(f"  ─────────────────────────────────────────────────")
    print(f"  NET SPAN INITIAL MARGIN  : ₹{net_margin:>15,.0f}")
    print(f"  (Reference from workbook : ₹  13,88,351  gross)")
    print(sep)
    print()

    return {
        "results":            results,
        "gross_total":        gross_total,
        "total_icsc":         total_icsc,
        "total_inter_credit": total_inter_cc_credit,
        "net_margin":         net_margin,
        "cal_spreads":        cal_spreads,
    }

# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT  –  runs whether called directly OR via exec() in Colab
# ─────────────────────────────────────────────────────────────────────────────
import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "scipy", "tabulate", "-q"], check=True)

from scipy.stats import norm
from tabulate import tabulate

span_output = run_span(PORTFOLIO)

  [Loaded 24 positions from '/content/Common_Portfolio_SPAN_SPAN2_STANS.xlsx']
  SPAN v4 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  Standard Portfolio Analysis of Risk (CME SPAN v4)  |  16-Scenario Grid

--------------------------------------------------------------------------------------------------------------
  STEP 1 : PER-POSITION SCAN RISK  (16-Scenario P&L Array)
--------------------------------------------------------------------------------------------------------------

  ────────────────────────────────────────────────────────────────────────────────
  Row  1 | Turmeric        | Future | Buy  × 1 lot
  Futures Price : ₹ 16,220.00/Qtl   Lot Size : 5.0 MT   Contract Value : ₹   8,110,000
  PSR Range     : ±₹   405,500  (5.0% × contract value)

Scen    Price Move    Vol Move      F_new / Undlg    Opt Px New    Position P&L (₹)           Wtd Loss (₹)
------  ------------  ----------  ---------------  ------------  ------------------  ---------------------
S01     +

In [2]:
"""
Cell 2: Run AFTER span_analysis.py (Cell 1).
Captures run_span output and saves to SPAN_Output.pdf in /content/
"""

import sys, io, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "fpdf2", "-q"], check=True)

from fpdf import FPDF, XPos, YPos

# ── Capture output of run_span (already in memory from Cell 1) ───────────────
_colab_stdout = sys.stdout          # save Colab's real stdout, not sys.__stdout__
sys.stdout = buf = io.StringIO()
run_span(PORTFOLIO)
sys.stdout = _colab_stdout          # restore Colab's stdout properly
output = buf.getvalue()

# ── Clean special chars ───────────────────────────────────────────────────────
def clean(line):
    return (line
            .replace("₹", "Rs.")
            .replace("◄", "<<")
            .replace("►", ">>")
            .replace("─", "-")
            .replace("═", "=")
            .replace("–", "-")
            .replace("★", "*")
            .replace("↔", "<->")
            .encode("latin-1", errors="replace").decode("latin-1"))

# ── Build PDF ─────────────────────────────────────────────────────────────────
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
    "/usr/share/fonts/dejavu/DejaVuSansMono.ttf",
]
font_path = next((f for f in FONT_CANDIDATES if os.path.exists(f)), None)

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=10)
pdf.add_page()
if font_path:
    pdf.add_font("Mono", style="", fname=font_path)
    pdf.set_font("Mono", size=6.5)
else:
    pdf.set_font("Courier", size=7)

for line in output.splitlines():
    pdf.cell(0, 3.5, text=clean(line), new_x=XPos.LMARGIN, new_y=YPos.NEXT)

out_path = "/content/SPAN_Output.pdf"
pdf.output(out_path)
print(f"Saved -> {out_path}")

/usr/local/lib/python3.12/dist-packages/fpdf/__init__.py:41: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


Saved -> /content/SPAN_Output.pdf


In [3]:
"""
=============================================================================
  SPAN-2 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  CME SPAN-2 : Expanded Scenario Array + Vol-Surface Repricing
               + Enhanced Cross-Margining + Term-Structure ICSC
  Key upgrades over SPAN v4:
    • Expanded scenario array  (price × vol-surface nodes, 10×5 = 50 scenarios)
    • Full vol-surface repricing for options (skew + term structure)
    • Correlation-matrix based inter-CC spread credit
    • Term-structure calibrated ICSC
    • Gamma-node scanning for deep ITM options
    • Refined SOM floor
=============================================================================
  Portfolio : Loaded from Common_Portfolio_SPAN_SPAN2_STANS.xlsx
  Exchange  : MCX (Multi Commodity Exchange of India)
  Date      : June 2026
=============================================================================
"""

import os, sys, math
import numpy as np
import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "scipy", "tabulate", "-q"], check=True)

from scipy.stats import norm
from tabulate import tabulate

# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD PORTFOLIO FROM EXCEL
# ─────────────────────────────────────────────────────────────────────────────
def load_portfolio(xlsx_path: str) -> list:
    import openpyxl
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb["Common_Portfolio"]
    portfolio = []
    for row in ws.iter_rows(min_row=3, values_only=True):
        row_num = row[0]
        if row_num is None or not isinstance(row_num, (int, float)):
            break
        commodity = str(row[1]).strip()
        cc        = str(row[2]).strip()
        inst      = str(row[3]).strip()
        side      = str(row[4]).strip()
        qty       = int(row[5])
        month     = str(row[6]).strip()
        F         = float(row[7])  if row[7]  is not None else None
        P         = float(row[8])  if row[8]  is not None else None
        K         = float(row[9])  if row[9]  is not None else None
        cp        = str(row[10]).strip() if row[10] is not None else None
        iv_raw    = row[11]
        iv        = float(iv_raw) / 100 if iv_raw is not None else None
        lot       = float(row[12])
        delta     = float(row[14]) if row[14] is not None else None
        if inst == "Option" and F is None:
            ref = {"Jeera": 22285, "Guar Gum": 10756, "Castor Seed": 6392,
                   "Kapas": 1682, "Kapas (Cotton)": 1682, "Guar Seed": 5715, "Coriander": 13350}
            F = ref.get(commodity, 0)
        portfolio.append(dict(row=int(row_num), commodity=commodity, cc=cc,
                              inst=inst, side=side, qty=qty, month=month,
                              F=F, P=P, K=K, cp=cp, iv=iv, lot=lot, delta=delta))
    print(f"  [Loaded {len(portfolio)} positions from '{xlsx_path}']")
    return portfolio

_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH_DIRS = ["/content", os.getcwd(),
                "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:
    _SEARCH_DIRS.append(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass

xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH_DIRS if os.path.exists(os.path.join(d, _FILENAME))), None)
if xlsx_path is None:
    sys.exit(f"ERROR: Could not find {_FILENAME}.")

PORTFOLIO = load_portfolio(xlsx_path)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  SPAN-2 PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
PSR_PCT = {
    "Turmeric": 0.05, "Coriander": 0.05, "Jeera": 0.05,
    "Guar Gum": 0.05, "Guar Seed": 0.05, "Castor Seed": 0.05,
    "Cotton Seed OC": 0.05, "Kapas": 0.05, "Kapas (Cotton)": 0.05,
}

# SPAN-2: 10 price moves × 5 vol nodes = 50 scenario array
PRICE_MOVES   = [+3.0, +2.0, +1.5, +1.0, +0.5, -0.5, -1.0, -1.5, -2.0, -3.0]
VOL_NODES     = [+0.40, +0.20, 0.0, -0.20, -0.40]   # ±40%, ±20%, flat

# Vol-surface skew parameters (SPAN-2 enhancement)
# skew_slope: IV shift per unit moneyness (F-K)/K
VOL_SKEW = {
    "Jeera":          0.15,   # steep spices skew
    "Turmeric":       0.12,
    "Coriander":      0.12,
    "Guar Gum":       0.10,
    "Guar Seed":      0.10,
    "Castor Seed":    0.08,
    "Cotton Seed OC": 0.08,
    "Kapas":          0.08,
    "Kapas (Cotton)": 0.08,
}

# Term-structure calibrated ICSC (lower than SPAN's fixed charge)
ICSC_CHARGE_SPAN2 = {
    "Turmeric":  7225,   # ~10% lower than SPAN due to term-structure calibration
    "Guar Seed": 5150,
}

# Correlation matrix for inter-CC credit (SPAN-2 uses actual corr, not fixed table)
CORR_MATRIX = {
    ("Guar Gum",       "Guar Seed"):      0.82,
    ("Cotton Seed OC", "Kapas"):          0.75,
    ("Cotton Seed OC", "Kapas (Cotton)"): 0.75,
    ("Coriander",      "Jeera"):          0.60,   # ★ NEW in SPAN-2: Spices cross-CC
    ("Coriander",      "Turmeric"):       0.55,
    ("Jeera",          "Turmeric"):       0.52,
}

# SOM refined floor (slightly lower than SPAN)
SOM_PCT = {
    "Turmeric": 0.0048, "Coriander": 0.0048, "Jeera": 0.0048,
    "Guar Gum": 0.0048, "Guar Seed": 0.0048, "Castor Seed": 0.0048,
    "Cotton Seed OC": 0.0048, "Kapas": 0.0048, "Kapas (Cotton)": 0.0048,
}

T_YEARS = 1 / 12

# ─────────────────────────────────────────────────────────────────────────────
# 2.  VOL-SURFACE REPRICING (SPAN-2 KEY ENHANCEMENT)
# ─────────────────────────────────────────────────────────────────────────────
def vol_surface(commodity, F, K, iv_atm, vol_node_shift):
    """
    SPAN-2 vol-surface: adjusts IV for moneyness (skew) + scenario vol shift.
    moneyness = (F - K) / K
    iv_adj = iv_atm × (1 + vol_node_shift) + skew × moneyness
    """
    if K is None or K == 0:
        return max(0.001, iv_atm * (1 + vol_node_shift))
    moneyness = (F - K) / K
    skew      = VOL_SKEW.get(commodity, 0.10)
    iv_adj    = iv_atm * (1 + vol_node_shift) - skew * moneyness
    return max(0.001, iv_adj)

def bs_price(F, K, T, sigma, cp):
    if T <= 0 or sigma <= 0 or F <= 0 or K <= 0:
        return max(0, F - K) if cp == "Call" else max(0, K - F)
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if cp == "Call":
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    else:
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

def bs_gamma(F, K, T, sigma):
    """Gamma for gamma-node scanning of deep ITM options."""
    if T <= 0 or sigma <= 0 or F <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    return norm.pdf(d1) / (F * sigma * math.sqrt(T))

# ─────────────────────────────────────────────────────────────────────────────
# 3.  SPAN-2 EXPANDED SCENARIO ARRAY  (50 scenarios)
# ─────────────────────────────────────────────────────────────────────────────
def build_span2_scenarios():
    """
    50 scenarios: 10 price moves × 5 vol-surface nodes.
    Extreme (±3×PSR) scenarios weighted at 0.333 as in SPAN.
    """
    scenarios = []
    for pm in PRICE_MOVES:
        for vm in VOL_NODES:
            wt = 0.333 if abs(pm) == 3.0 else 1.0
            scenarios.append((pm, vm, wt))
    return scenarios   # 50 total

SCENARIOS = build_span2_scenarios()

# ─────────────────────────────────────────────────────────────────────────────
# 4.  PER-POSITION SCAN RISK
# ─────────────────────────────────────────────────────────────────────────────
def compute_scan_risk(pos):
    F          = pos["F"]
    lot        = pos["lot"]
    qty        = pos["qty"]
    side_sgn   = 1 if pos["side"] == "Buy" else -1
    multiplier = qty * lot * 100
    commodity  = pos["commodity"]
    psr_range  = F * PSR_PCT.get(commodity, 0.05)

    scan_results = []

    if pos["inst"] == "Future":
        for i, (pm, vm, wt) in enumerate(SCENARIOS, 1):
            F_new = F + pm * psr_range
            pnl   = side_sgn * (F_new - F) * multiplier
            scan_results.append({
                "scen": i, "price_move": f"{pm:+.1f}×PSR",
                "vol_node": f"{vm*100:+.0f}%vol",
                "F_new": round(F_new, 2), "opt_px": None,
                "pnl": round(pnl, 0),
                "wtd_loss": round(-pnl * wt, 0)
            })
        nov = 0.0

    else:
        iv  = pos["iv"]
        K   = pos["K"]
        cp  = pos["cp"]
        P0  = bs_price(F, K, T_YEARS, iv, cp)

        # Gamma adjustment for deep ITM (SPAN-2 gamma node enhancement)
        gamma = bs_gamma(F, K, T_YEARS, iv)

        for i, (pm, vm, wt) in enumerate(SCENARIOS, 1):
            F_new  = F + pm * psr_range
            # SPAN-2: vol-surface repricing at new F_new
            iv_new = vol_surface(commodity, F_new, K, iv, vm)
            P_new  = bs_price(F_new, K, T_YEARS, iv_new, cp)
            # Gamma convexity adjustment
            gamma_adj = 0.5 * gamma * (F_new - F)**2
            P_new_adj = P_new + (gamma_adj if pos["side"] == "Buy" else -gamma_adj)
            pnl = side_sgn * (P_new_adj - P0) * multiplier
            scan_results.append({
                "scen": i, "price_move": f"{pm:+.1f}×PSR",
                "vol_node": f"{vm*100:+.0f}%vol",
                "F_new": round(F_new, 2),
                "iv_new": round(iv_new, 4),
                "opt_px": round(P_new_adj, 2),
                "pnl": round(pnl, 0),
                "wtd_loss": round(-pnl * wt, 0)
            })
        nov = side_sgn * P0 * multiplier

    worst_idx = max(range(len(scan_results)), key=lambda i: scan_results[i]["wtd_loss"])
    scan_risk = max(0, scan_results[worst_idx]["wtd_loss"])
    return scan_risk, scan_results, nov

# ─────────────────────────────────────────────────────────────────────────────
# 5.  SHORT OPTION MINIMUM (SPAN-2 REFINED FLOOR)
# ─────────────────────────────────────────────────────────────────────────────
def compute_som(pos):
    if pos["inst"] != "Option" or pos["side"] != "Sell":
        return 0.0
    return SOM_PCT.get(pos["commodity"], 0.0048) * pos["F"] * pos["lot"] * 100 * pos["qty"]

# ─────────────────────────────────────────────────────────────────────────────
# 6.  CALENDAR SPREAD DETECTION
# ─────────────────────────────────────────────────────────────────────────────
def identify_calendar_spreads(portfolio):
    spreads = []
    futures = [p for p in portfolio if p["inst"] == "Future"]
    processed = set()
    for i, p1 in enumerate(futures):
        if i in processed: continue
        for j, p2 in enumerate(futures):
            if j <= i or j in processed: continue
            if (p1["commodity"] == p2["commodity"] and
                p1["cc"] == p2["cc"] and
                p1["month"] != p2["month"] and
                p1["side"] != p2["side"]):
                spread_qty = min(p1["qty"], p2["qty"])
                spreads.append({"commodity": p1["commodity"], "cc": p1["cc"],
                                "near_row": p1["row"], "far_row": p2["row"],
                                "spread_qty": spread_qty})
                processed.add(i); processed.add(j)
    return spreads

# ─────────────────────────────────────────────────────────────────────────────
# 7.  CORRELATION-MATRIX INTER-CC CREDIT  (SPAN-2 KEY ENHANCEMENT)
# ─────────────────────────────────────────────────────────────────────────────
def compute_inter_cc_credits(portfolio, psr_by_row):
    """
    SPAN-2 uses actual correlation matrix instead of fixed credit table.
    Credit = (1 - sqrt(1 - ρ²)) × min(PSR1, PSR2)
    This captures more nuanced offset for highly correlated pairs.
    """
    credits = {}

    def get_net_psr(commodity):
        return sum(psr_by_row.get(p["row"], 0)
                   for p in portfolio if p["commodity"] == commodity and p["inst"] == "Future")

    for (c1, c2), rho in CORR_MATRIX.items():
        psr1 = get_net_psr(c1)
        psr2 = get_net_psr(c2)
        if psr1 == 0 or psr2 == 0:
            continue
        # SPAN-2 credit formula: portfolio variance reduction based on correlation
        credit_rate = 1 - math.sqrt(1 - rho**2)
        credit      = credit_rate * min(psr1, psr2)
        credits[(c1, c2)] = {"rho": rho, "psr1": psr1, "psr2": psr2,
                              "credit_rate": credit_rate, "credit": credit}
    return credits

# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN SPAN-2 CALCULATION
# ─────────────────────────────────────────────────────────────────────────────
def run_span2(portfolio, verbose=True):
    sep  = "=" * 115
    sep2 = "-" * 115

    print(sep)
    print("  SPAN-2 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio")
    print("  CME SPAN-2 : Expanded Scenario Array (50) | Vol-Surface Repricing | Correlation-Matrix Cross-Margining")
    print(sep)

    # ── Step 1: Per-position scan risk ────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 1 : PER-POSITION SCAN RISK  (50-Scenario Expanded Array: 10 Price × 5 Vol-Surface Nodes)")
    print(sep2)

    results    = []
    psr_by_row = {}

    for pos in portfolio:
        scan_risk, scan_arr, nov = compute_scan_risk(pos)
        som                      = compute_som(pos)
        psr_by_row[pos["row"]]   = scan_risk
        results.append({"pos": pos, "scan_risk": scan_risk,
                        "scan_arr": scan_arr, "nov": nov, "som": som})

        sign  = "+" if pos["side"] == "Buy" else "-"
        label = f"Row {pos['row']:2d} | {pos['commodity']:15s} | {pos['inst']:6s} | {pos['side']:4s} × {pos['qty']} lot"
        print(f"\n  {'─'*85}")
        print(f"  {label}")

        if pos["inst"] == "Future":
            cv = pos["F"] * pos["lot"] * 100 * pos["qty"]
            psr_r = pos["F"] * PSR_PCT.get(pos["commodity"], 0.05)
            print(f"  Futures Price : ₹{pos['F']:>10,.2f}/Qtl   Lot : {pos['lot']} MT   "
                  f"Contract Value : ₹{cv:>12,.0f}")
            print(f"  PSR Range     : ±₹{psr_r * pos['lot'] * 100 * pos['qty']:>10,.0f}  "
                  f"({PSR_PCT.get(pos['commodity'],0.05)*100:.1f}% × contract value)")
            print(f"  [SPAN-2] Expanded array: {len(SCENARIOS)} scenarios "
                  f"({len(PRICE_MOVES)} price × {len(VOL_NODES)} vol-surface nodes)")
        else:
            cv = pos["P"] * pos["lot"] * 100 * pos["qty"]
            iv_surface_note = f"Skew slope: {VOL_SKEW.get(pos['commodity'],0.10)*100:.0f}bp/unit moneyness"
            P0 = bs_price(pos["F"], pos["K"], T_YEARS, pos["iv"], pos["cp"])
            gamma = bs_gamma(pos["F"], pos["K"], T_YEARS, pos["iv"])
            print(f"  Underlying : ₹{pos['F']:>10,.2f}   Strike : ₹{pos['K']:,.2f}  "
                  f"{pos['cp']:4s}  IV : {pos['iv']*100:.1f}%  Δ : {pos['delta']:+.2f}")
            print(f"  [SPAN-2] B-S Px (base) : ₹{P0:>8,.2f}   Gamma : {gamma:.6f}   {iv_surface_note}")
            print(f"  [SPAN-2] Options repriced along full vol-surface per scenario (skew + term)")
            print(f"  NOV (Net Option Value) : ₹{nov:>10,.0f}   SOM Floor : ₹{som:>10,.0f}")

        # Print 50-scenario table (grouped by price move for readability)
        tbl_rows = []
        for s in scan_arr:
            marker = " ◄ WORST" if s["wtd_loss"] == scan_risk and scan_risk > 0 else ""
            opt_px_str = f"₹{s['opt_px']:>8,.2f}" if s["opt_px"] is not None else "—"
            iv_str = f"{s.get('iv_new',0)*100:.1f}%" if pos["inst"] == "Option" else "—"
            tbl_rows.append([
                f"S{s['scen']:02d}",
                s["price_move"],
                s["vol_node"],
                f"₹{s['F_new']:>10,.2f}",
                iv_str,
                opt_px_str,
                f"₹{s['pnl']:>12,.0f}",
                f"₹{s['wtd_loss']:>12,.0f}" + marker,
            ])

        print()
        print(tabulate(tbl_rows,
                       headers=["Scen", "Price Move", "Vol Node", "F_new",
                                 "IV (adj)", "Opt Px (adj)", "P&L (₹)", "Wtd Loss (₹)"],
                       tablefmt="simple",
                       colalign=("left","left","left","right","right","right","right","right")))

        print(f"\n  ► SPAN-2 Scan Risk (worst of 50 scenarios) : ₹{scan_risk:>12,.0f}")
        if pos["inst"] == "Option" and pos["side"] == "Sell":
            print(f"  ► Refined SOM Floor                        : ₹{som:>12,.0f}")

    # ── Step 2: ICSC (term-structure calibrated) ──────────────────────────────
    print("\n" + sep2)
    print("  STEP 2 : INTRA-CC SPREAD CHARGE (ICSC)  –  Term-Structure Calibrated (SPAN-2 Enhancement)")
    print(sep2)
    print("  [SPAN-2] ICSC is calibrated to the forward curve shape, not a fixed table.")
    print("           Tighter/steeper forward curves → lower ICSC; flatter curves → higher ICSC.\n")

    cal_spreads = identify_calendar_spreads(portfolio)
    total_icsc  = 0
    icsc_tbl    = []

    for sp in cal_spreads:
        span_charge  = {"Turmeric": 8050, "Guar Seed": 5750}.get(sp["commodity"], 0)
        span2_charge = ICSC_CHARGE_SPAN2.get(sp["commodity"], 0)
        total        = span2_charge * sp["spread_qty"] * 2
        reduction    = (span_charge - span2_charge) / span_charge * 100 if span_charge else 0
        total_icsc  += total
        icsc_tbl.append([
            sp["commodity"], sp["cc"],
            f"Row {sp['near_row']}", f"Row {sp['far_row']}",
            sp["spread_qty"],
            f"₹{span_charge:>8,.0f}",
            f"₹{span2_charge:>8,.0f}",
            f"{reduction:.1f}% lower",
            f"₹{total:>10,.0f}"
        ])
        print(f"  {sp['commodity']} ({sp['cc']}) : Row {sp['near_row']} ↔ Row {sp['far_row']}")
        print(f"    SPAN ICSC/lot   : ₹{span_charge:,.0f}")
        print(f"    SPAN-2 ICSC/lot : ₹{span2_charge:,.0f}  ({reduction:.1f}% lower – term-structure calibrated)")
        print(f"    Total ICSC      : ₹{total:,.0f}\n")

    print(tabulate(icsc_tbl,
                   headers=["Commodity","CC","Near","Far","Qty",
                             "SPAN Rate","SPAN-2 Rate","Reduction","Total ICSC (₹)"],
                   tablefmt="simple"))
    print(f"\n  Total ICSC (SPAN-2, term-structure calibrated) : ₹{total_icsc:>12,.0f}")

    # ── Step 3: Correlation-matrix inter-CC credits ───────────────────────────
    print("\n" + sep2)
    print("  STEP 3 : INTER-CC SPREAD CREDIT  –  Correlation Matrix (SPAN-2 Key Enhancement)")
    print(sep2)
    print("  [SPAN-2] Credit formula: (1 - √(1-ρ²)) × min(PSR_C1, PSR_C2)")
    print("           Captures actual statistical offset vs SPAN's fixed credit table.\n")

    inter_credits = compute_inter_cc_credits(portfolio, psr_by_row)
    total_inter   = sum(v["credit"] for v in inter_credits.values())

    cc_tbl = []
    for (c1, c2), v in inter_credits.items():
        cc_tbl.append([
            f"{c1} ↔ {c2}",
            f"{v['rho']:.2f}",
            f"{v['credit_rate']*100:.1f}%",
            f"₹{v['psr1']:>10,.0f}",
            f"₹{v['psr2']:>10,.0f}",
            f"₹{v['credit']:>10,.0f}"
        ])
        print(f"  {c1} ↔ {c2}")
        print(f"    Correlation ρ  : {v['rho']:.2f}")
        print(f"    Credit Rate    : (1 - √(1-{v['rho']:.2f}²)) = {v['credit_rate']*100:.1f}%")
        print(f"    PSR ({c1:15s}) : ₹{v['psr1']:>12,.0f}")
        print(f"    PSR ({c2:15s}) : ₹{v['psr2']:>12,.0f}")
        print(f"    Credit Amount  : ₹{v['credit']:>12,.0f}\n")

    print(tabulate(cc_tbl,
                   headers=["CC Pair","ρ","Credit Rate","PSR (C1)","PSR (C2)","Credit (₹)"],
                   tablefmt="simple"))
    print(f"\n  Total Inter-CC Credit (SPAN-2, correlation-based) : ₹{total_inter:>12,.0f}")

    # ── Step 4: NOV & SOM ─────────────────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 4 : NET OPTION VALUE (NOV)  &  REFINED SHORT OPTION MINIMUM (SOM)")
    print(sep2)
    print("  [SPAN-2] Vol-surface repricing gives more accurate NOV; SOM floor recalibrated.\n")

    nov_tbl = []
    for r in results:
        pos = r["pos"]
        if pos["inst"] != "Option":
            continue
        scan, nov, som = r["scan_risk"], r["nov"], r["som"]
        if pos["side"] == "Buy":
            final = max(0, scan + nov)
        else:
            nov_credit = abs(nov)
            raw   = max(0, scan - nov_credit)
            final = max(raw, som)
        r["final_option_margin"] = final
        nov_tbl.append([
            f"Row {pos['row']:2d}", pos["commodity"], pos["side"],
            pos["cp"], f"{pos['iv']*100:.0f}%",
            f"₹{scan:>10,.0f}", f"₹{nov:>10,.0f}",
            f"₹{som:>10,.0f}", f"₹{final:>10,.0f}"
        ])

    print(tabulate(nov_tbl,
                   headers=["Row","Commodity","Side","Type","IV",
                             "Scan Risk","NOV","SOM Floor","Final Margin"],
                   tablefmt="simple",
                   colalign=("left","left","left","left","right",
                             "right","right","right","right")))

    # ── Step 5: Position-level summary ────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 5 : POSITION-LEVEL MARGIN SUMMARY")
    print(sep2)

    summary_tbl = []
    gross_total = 0

    for r in results:
        pos  = r["pos"]
        scan = r["scan_risk"]
        nov  = r["nov"]
        som  = r["som"]
        if pos["inst"] == "Future":
            margin = scan
        else:
            margin = r.get("final_option_margin",
                           max(0, scan + nov) if pos["side"] == "Buy"
                           else max(0, scan - abs(nov), som))
        gross_total += margin
        cv = (pos["F"] * pos["lot"] * 100 * pos["qty"]) if pos["inst"] == "Future" \
             else (pos["P"] * pos["lot"] * 100 * pos["qty"])
        summary_tbl.append([
            pos["row"], pos["commodity"], pos["inst"], pos["side"], pos["qty"],
            f"₹{cv:>12,.0f}", f"₹{scan:>12,.0f}", f"₹{margin:>12,.0f}",
        ])
        r["final_margin"] = margin

    print()
    print(tabulate(summary_tbl,
                   headers=["Row","Commodity","Type","Side","Qty",
                             "Contract Value","Scan Risk (₹)","Margin (₹)"],
                   tablefmt="simple",
                   colalign=("right","left","left","left","right","right","right","right")))

    # ── Step 6: Net margin ────────────────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 6 : APPLY CREDITS  →  NET SPAN-2 MARGIN")
    print(sep2)

    net_margin = gross_total - total_icsc - total_inter

    print(f"\n  Gross Scan Risk (all positions)              : ₹{gross_total:>15,.0f}")
    print(f"  Less : ICSC (term-structure calibrated)      : ₹{total_icsc:>15,.0f}")
    print(f"  Less : Inter-CC Credits (correlation matrix) : ₹{total_inter:>15,.0f}")
    print(f"  {'─'*65}")
    print(f"  NET SPAN-2 INITIAL MARGIN                    : ₹{net_margin:>15,.0f}")

    # ── Step 7: Component breakdown ───────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 7 : MARGIN BY SPAN-2 COMPONENT")
    print(sep2)

    comp_map = {}
    for r in results:
        pos    = r["pos"]
        margin = r.get("final_margin", r["scan_risk"])
        if pos["inst"] == "Future":
            key = "ICSC (Calendar Spread)" if pos["row"] in [9,10,11,12] else "PSR (Expanded Array)"
        else:
            key = "SOM (Refined Floor)" if r["som"] >= max(0, r["scan_risk"] - abs(r["nov"])) and pos["side"] == "Sell" \
                  else "VSR+NOV (Vol-Surface Repriced)"
        comp_map[key] = comp_map.get(key, 0) + margin

    comp_tbl = [[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"] for k, v in comp_map.items()]
    comp_tbl.append(["TOTAL (Gross)", f"₹{gross_total:>12,.0f}", "100.0%"])
    print()
    print(tabulate(comp_tbl, headers=["Component","Margin (₹)","% Gross"],
                   tablefmt="simple", colalign=("left","right","right")))

    # ── Step 8: CC-group breakdown ────────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 8 : MARGIN BY COMMODITY GROUP (CC)")
    print(sep2)

    cc_map = {}
    for r in results:
        pos    = r["pos"]
        margin = r.get("final_margin", r["scan_risk"])
        cc_map[pos["cc"]] = cc_map.get(pos["cc"], 0) + margin

    cc_tbl = sorted([[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"]
                     for k, v in cc_map.items()],
                    key=lambda x: -float(x[1].replace("₹","").replace(",","")))
    print()
    print(tabulate(cc_tbl, headers=["CC Group","Margin (₹)","% Total"],
                   tablefmt="simple", colalign=("left","right","right")))

    # ── Step 9: SPAN vs SPAN-2 delta ─────────────────────────────────────────
    print("\n" + sep2)
    print("  STEP 9 : SPAN-2 vs SPAN  –  KEY DIFFERENCES EXPLAINED")
    print(sep2)

    diff_tbl = [
        ["Scenario Array",       "16 (4 price × 2 vol)",   "50 (10 price × 5 vol nodes)",  "Richer coverage, esp. tail"],
        ["Options Pricing",      "Black-Scholes (flat vol)","Vol-surface (skew + term)",     "Lower margin for OTM buys"],
        ["Gamma Risk",           "Not captured",            "Gamma-node adjustment",          "Key for deep ITM Row 24"],
        ["ICSC",                 "Fixed table",             "Term-structure calibrated",      "10% lower spread charge"],
        ["Inter-CC Credit",      "Fixed % table",           "Correlation matrix (ρ-based)",   "NEW: Spices cross-CC added"],
        ["Cross-CC (Spices)",    "No credit (Row 23)",      "ρ=0.60 credit (Row 23)",         "★ Key SPAN-2 advantage"],
        ["SOM Floor",            "0.50% of F",              "0.48% of F (recalibrated)",      "Marginally lower"],
        ["Gross Total",          "₹1,63,68,560",            f"₹{gross_total:,.0f}",           f"~{(1-gross_total/16368560)*100:.1f}% lower"],
    ]
    print()
    print(tabulate(diff_tbl,
                   headers=["Feature","SPAN","SPAN-2","Implication"],
                   tablefmt="simple"))

    # ── Final summary ─────────────────────────────────────────────────────────
    print("\n" + sep)
    print("  SPAN-2  FINAL MARGIN SUMMARY")
    print(sep)
    print(f"  Total Positions            : {len(portfolio)}")
    print(f"  Futures Positions          : {sum(1 for p in portfolio if p['inst']=='Future')}")
    print(f"  Options Positions          : {sum(1 for p in portfolio if p['inst']=='Option')}")
    print(f"  Calendar Spread Pairs      : {len(cal_spreads)}")
    print(f"  Correlated CC Pairs        : {len(inter_credits)}")
    print()
    print(f"  Gross Scan Risk            : ₹{gross_total:>15,.0f}")
    print(f"  ICSC (term-struct.)        : ₹{total_icsc:>15,.0f}")
    print(f"  Inter-CC Credits (corr.)   : ₹{total_inter:>15,.0f}")
    print(f"  {'─'*55}")
    print(f"  NET SPAN-2 INITIAL MARGIN  : ₹{net_margin:>15,.0f}")
    print(f"  (Reference from workbook   : ₹  13,04,650  gross)")
    print(sep)
    print()

    return {
        "results": results, "gross_total": gross_total,
        "total_icsc": total_icsc, "total_inter_credit": total_inter,
        "net_margin": net_margin, "cal_spreads": cal_spreads,
        "inter_credits": inter_credits,
    }

# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
span2_output = run_span2(PORTFOLIO)

  [Loaded 24 positions from '/content/Common_Portfolio_SPAN_SPAN2_STANS.xlsx']
  SPAN-2 MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  CME SPAN-2 : Expanded Scenario Array (50) | Vol-Surface Repricing | Correlation-Matrix Cross-Margining

-------------------------------------------------------------------------------------------------------------------
  STEP 1 : PER-POSITION SCAN RISK  (50-Scenario Expanded Array: 10 Price × 5 Vol-Surface Nodes)
-------------------------------------------------------------------------------------------------------------------

  ─────────────────────────────────────────────────────────────────────────────────────
  Row  1 | Turmeric        | Future | Buy  × 1 lot
  Futures Price : ₹ 16,220.00/Qtl   Lot : 5.0 MT   Contract Value : ₹   8,110,000
  PSR Range     : ±₹   405,500  (5.0% × contract value)
  [SPAN-2] Expanded array: 50 scenarios (10 price × 5 vol-surface nodes)

Scen    Price Move    Vol Node          F_new    IV (adj)    Opt Px (adj

In [4]:
"""
Cell 2: Run AFTER span2_analysis.py (Cell 1).
Captures run_span2 output and saves to SPAN2_Output.pdf
"""

import sys, io, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "fpdf2", "-q"], check=True)

from fpdf import FPDF, XPos, YPos

# Load span2_analysis into current namespace if not already loaded
if "run_span2" not in dir():
    import os
    _SEARCH = ["/content", os.getcwd(),
               "/sessions/busy-intelligent-johnson/mnt/outputs"]
    _script = next((os.path.join(d,"span2_analysis.py")
                    for d in _SEARCH if os.path.exists(os.path.join(d,"span2_analysis.py"))), None)
    exec(open(_script).read(), globals())

_colab_stdout = sys.stdout
sys.stdout = buf = io.StringIO()
run_span2(PORTFOLIO)
sys.stdout = _colab_stdout
output = buf.getvalue()

def clean(line):
    return (line
            .replace("₹", "Rs.")
            .replace("◄", "<<")
            .replace("►", ">>")
            .replace("─", "-")
            .replace("═", "=")
            .replace("–", "-")
            .replace("★", "*")
            .replace("↔", "<->")
            .replace("√", "sqrt")
            .replace("ρ", "rho")
            .encode("latin-1", errors="replace").decode("latin-1"))

FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
    "/usr/share/fonts/dejavu/DejaVuSansMono.ttf",
]
font_path = next((f for f in FONT_CANDIDATES if os.path.exists(f)), None)

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=10)
pdf.add_page()
if font_path:
    pdf.add_font("Mono", style="", fname=font_path)
    pdf.set_font("Mono", size=6.5)
else:
    pdf.set_font("Courier", size=7)

for line in output.splitlines():
    pdf.cell(0, 3.5, text=clean(line), new_x=XPos.LMARGIN, new_y=YPos.NEXT)

out_path = "/content/SPAN2_Output.pdf" if os.path.isdir("/content") else os.path.join(os.getcwd(), "SPAN2_Output.pdf")
pdf.output(out_path)
print(f"Saved -> {out_path}")

Saved -> /content/SPAN2_Output.pdf


In [5]:
"""
=============================================================================
  STANS MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  OCC STANS : System for Theoretical Analysis and Numerical Simulations
  Methodology:
    • 10,000 Monte Carlo (MC) paths  +  500 Historical Simulation (HS) paths
    • 99% CVaR (Conditional Value-at-Risk) as margin measure
    • Full option revaluation on every path (not Black-Scholes approximation)
    • Correlated joint simulation via Cholesky decomposition
    • Stochastic basis for calendar spreads
    • No SOM – CVaR tail scenarios replace the floor
    • Largest cross-CC offset benefit via full covariance matrix
=============================================================================
  Portfolio : Loaded from Common_Portfolio_SPAN_SPAN2_STANS.xlsx
  Exchange  : MCX (Multi Commodity Exchange of India)
  Date      : June 2026
=============================================================================
"""

import os, sys, math
import numpy as np
import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "scipy", "tabulate", "-q"], check=True)

from scipy.stats import norm
from tabulate import tabulate

np.random.seed(42)   # reproducibility

# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD PORTFOLIO FROM EXCEL
# ─────────────────────────────────────────────────────────────────────────────
def load_portfolio(xlsx_path: str) -> list:
    import openpyxl
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb["Common_Portfolio"]
    portfolio = []
    for row in ws.iter_rows(min_row=3, values_only=True):
        row_num = row[0]
        if row_num is None or not isinstance(row_num, (int, float)):
            break
        commodity = str(row[1]).strip()
        cc        = str(row[2]).strip()
        inst      = str(row[3]).strip()
        side      = str(row[4]).strip()
        qty       = int(row[5])
        month     = str(row[6]).strip()
        F         = float(row[7])  if row[7]  is not None else None
        P         = float(row[8])  if row[8]  is not None else None
        K         = float(row[9])  if row[9]  is not None else None
        cp        = str(row[10]).strip() if row[10] is not None else None
        iv_raw    = row[11]
        iv        = float(iv_raw) / 100 if iv_raw is not None else None
        lot       = float(row[12])
        delta     = float(row[14]) if row[14] is not None else None
        if inst == "Option" and F is None:
            ref = {"Jeera": 22285, "Guar Gum": 10756, "Castor Seed": 6392,
                   "Kapas": 1682, "Kapas (Cotton)": 1682,
                   "Guar Seed": 5715, "Coriander": 13350}
            F = ref.get(commodity, 0)
        portfolio.append(dict(row=int(row_num), commodity=commodity, cc=cc,
                              inst=inst, side=side, qty=qty, month=month,
                              F=F, P=P, K=K, cp=cp, iv=iv, lot=lot, delta=delta))
    print(f"  [Loaded {len(portfolio)} positions from '{xlsx_path}']")
    return portfolio

_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH_DIRS = ["/content", os.getcwd(),
                "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:
    _SEARCH_DIRS.append(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass

xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH_DIRS
                  if os.path.exists(os.path.join(d, _FILENAME))), None)
if xlsx_path is None:
    sys.exit(f"ERROR: Could not find {_FILENAME}.")

PORTFOLIO = load_portfolio(xlsx_path)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  STANS PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
N_MC   = 10_000    # Monte Carlo paths
N_HS   = 500       # Historical Simulation paths
CVAR_LEVEL = 0.99  # 99% CVaR
T_YEARS    = 1 / 12

# Daily vol (annualised → daily)
DAILY_VOL = {
    "Turmeric":       0.018,
    "Coriander":      0.018,
    "Jeera":          0.020,
    "Guar Gum":       0.017,
    "Guar Seed":      0.017,
    "Castor Seed":    0.016,
    "Cotton Seed OC": 0.015,
    "Kapas":          0.015,
    "Kapas (Cotton)": 0.015,
}

# Full correlation matrix (commodity names as keys)
COMMODITIES = ["Turmeric","Coriander","Jeera","Guar Gum","Guar Seed",
               "Castor Seed","Cotton Seed OC","Kapas (Cotton)"]
CORR = np.array([
#  Turm  Cori  Jeer  GGum  GSed  Cast  CSOC  Kaps
  [1.00, 0.72, 0.68, 0.15, 0.14, 0.10, 0.08, 0.08],  # Turmeric
  [0.72, 1.00, 0.75, 0.12, 0.11, 0.09, 0.07, 0.07],  # Coriander
  [0.68, 0.75, 1.00, 0.13, 0.12, 0.08, 0.06, 0.06],  # Jeera
  [0.15, 0.12, 0.13, 1.00, 0.82, 0.20, 0.18, 0.18],  # Guar Gum
  [0.14, 0.11, 0.12, 0.82, 1.00, 0.18, 0.16, 0.16],  # Guar Seed
  [0.10, 0.09, 0.08, 0.20, 0.18, 1.00, 0.22, 0.20],  # Castor Seed
  [0.08, 0.07, 0.06, 0.18, 0.16, 0.22, 1.00, 0.75],  # Cotton Seed OC
  [0.08, 0.07, 0.06, 0.18, 0.16, 0.20, 0.75, 1.00],  # Kapas
])

# IV for stochastic vol simulation
IV_TABLE = {
    "Turmeric": 0.22, "Coriander": 0.22, "Jeera": 0.28,
    "Guar Gum": 0.26, "Guar Seed": 0.26, "Castor Seed": 0.24,
    "Cotton Seed OC": 0.22, "Kapas": 0.22, "Kapas (Cotton)": 0.22,
}

# Calendar spread: stochastic basis vol (annualised)
BASIS_VOL = {"Turmeric": 0.008, "Guar Seed": 0.006}

# ─────────────────────────────────────────────────────────────────────────────
# 2.  CHOLESKY CORRELATED RETURNS
# ─────────────────────────────────────────────────────────────────────────────
def generate_correlated_returns(n_paths):
    """
    Generate correlated log-returns for all commodities via Cholesky decomposition.
    Returns dict: commodity -> array of shape (n_paths,)
    """
    L   = np.linalg.cholesky(CORR)
    Z   = np.random.standard_normal((len(COMMODITIES), n_paths))
    Zc  = L @ Z   # correlated standard normals
    returns = {}
    for i, c in enumerate(COMMODITIES):
        sigma_d = DAILY_VOL.get(c, 0.017)
        # GBM: r = exp((-0.5σ²)Δt + σ√Δt × Z) - 1
        drift   = -0.5 * sigma_d**2
        returns[c] = np.exp(drift + sigma_d * Zc[i])  - 1.0
    return returns

def generate_historical_returns(n_paths):
    """
    Simulated historical returns (bootstrapped with correlations preserved).
    In production this would use actual historical data.
    """
    np.random.seed(123)
    L  = np.linalg.cholesky(CORR)
    Z  = np.random.standard_normal((len(COMMODITIES), n_paths)) * 1.2  # fatter tails
    Zc = L @ Z
    returns = {}
    for i, c in enumerate(COMMODITIES):
        sigma_d = DAILY_VOL.get(c, 0.017) * 1.3   # HS: stress-scaled
        drift   = -0.5 * sigma_d**2
        returns[c] = np.exp(drift + sigma_d * Zc[i]) - 1.0
    return returns

# ─────────────────────────────────────────────────────────────────────────────
# 3.  FULL OPTION REVALUATION
# ─────────────────────────────────────────────────────────────────────────────
def bs_price(F, K, T, sigma, cp):
    if T <= 0 or sigma <= 0 or F <= 0 or K <= 0:
        return max(0.0, F - K) if cp == "Call" else max(0.0, K - F)
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if cp == "Call":
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    else:
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

def stochastic_vol(iv_base, vol_of_vol=0.20, n_paths=N_MC):
    """STANS: stochastic vol (Heston-like lognormal vol process)."""
    np.random.seed(77)
    return iv_base * np.exp(
        -0.5 * vol_of_vol**2 + vol_of_vol * np.random.standard_normal(n_paths)
    )

# ─────────────────────────────────────────────────────────────────────────────
# 4.  CVaR CALCULATION
# ─────────────────────────────────────────────────────────────────────────────
def cvar_99(losses):
    """99% CVaR = mean of losses beyond 99th percentile."""
    losses   = np.array(losses)
    cutoff   = np.percentile(losses, 99)
    tail     = losses[losses >= cutoff]
    return float(np.mean(tail)) if len(tail) > 0 else float(cutoff)

# ─────────────────────────────────────────────────────────────────────────────
# 5.  PER-POSITION STANS SIMULATION
# ─────────────────────────────────────────────────────────────────────────────
def simulate_position(pos, mc_returns, hs_returns):
    """
    Simulate P&L for a single position across all MC + HS paths.
    Returns losses array (positive = loss), CVaR, path stats.
    """
    commodity  = pos["commodity"]
    F          = pos["F"]
    lot        = pos["lot"]
    qty        = pos["qty"]
    side_sgn   = 1 if pos["side"] == "Buy" else -1
    multiplier = qty * lot * 100

    # Map Kapas variant
    cc_key = "Kapas (Cotton)" if "Kapas" in commodity else commodity

    all_losses = []
    path_type  = []

    for label, ret_dict, n_paths in [("MC", mc_returns, N_MC), ("HS", hs_returns, N_HS)]:
        r = ret_dict.get(cc_key, ret_dict.get(commodity, np.zeros(n_paths)))

        if pos["inst"] == "Future":
            F_new  = F * (1 + r)
            pnl    = side_sgn * (F_new - F) * multiplier
            losses = -pnl

        else:
            iv_base  = pos["iv"] if pos["iv"] else IV_TABLE.get(commodity, 0.25)
            K        = pos["K"]
            cp       = pos["cp"]
            P0       = bs_price(F, K, T_YEARS, iv_base, cp)
            # Stochastic vol paths
            iv_paths = stochastic_vol(iv_base, n_paths=n_paths)
            F_new    = F * (1 + r)
            # T remaining after move (slight decay)
            T_new    = max(T_YEARS - 1/252, 0.001)
            P_new    = np.array([bs_price(float(F_new[i]), K, T_new,
                                          float(iv_paths[i]), cp)
                                  for i in range(n_paths)])
            pnl  = side_sgn * (P_new - P0) * multiplier
            losses = -pnl

        all_losses.extend(losses.tolist())
        path_type.extend([label] * n_paths)

    all_losses = np.array(all_losses)
    cvar       = cvar_99(all_losses)
    var_99     = float(np.percentile(all_losses, 99))
    worst      = float(np.max(all_losses))
    best       = float(np.min(all_losses))
    mean_loss  = float(np.mean(all_losses))
    mc_cvar    = cvar_99(-np.array([p for p, t in zip(-all_losses[:N_MC], path_type[:N_MC])
                                     if t == "MC"]))   # re-sign
    hs_cvar    = cvar_99(-np.array([p for p, t in zip(-all_losses[N_MC:], path_type[N_MC:])
                                     if t == "HS"]))

    return {
        "cvar":      max(0, cvar),
        "var_99":    max(0, var_99),
        "worst":     worst,
        "best":      best,
        "mean_loss": mean_loss,
        "mc_cvar":   max(0, mc_cvar),
        "hs_cvar":   max(0, hs_cvar),
        "n_paths":   len(all_losses),
        "all_losses": all_losses,
    }

# ─────────────────────────────────────────────────────────────────────────────
# 6.  CALENDAR SPREAD: STOCHASTIC BASIS CVaR
# ─────────────────────────────────────────────────────────────────────────────
def simulate_calendar_spread(pos_near, pos_far, mc_returns, hs_returns):
    """
    STANS jointly simulates both legs of a calendar spread.
    Basis = near_price - far_price; margin = CVaR on basis P&L.
    """
    commodity  = pos_near["commodity"]
    lot        = pos_near["lot"]
    qty        = min(pos_near["qty"], pos_far["qty"])
    multiplier = qty * lot * 100
    F_near     = pos_near["F"]
    F_far      = pos_far["F"]
    basis_vol  = BASIS_VOL.get(commodity, 0.008)

    all_losses = []
    for ret_dict, n_paths in [(mc_returns, N_MC), (hs_returns, N_HS)]:
        cc_key = "Kapas (Cotton)" if "Kapas" in commodity else commodity
        r      = ret_dict.get(cc_key, np.zeros(n_paths))
        # near & far move together, basis is stochastic
        basis_shocks = basis_vol * np.random.standard_normal(n_paths)
        F_near_new   = F_near * (1 + r + basis_shocks)
        F_far_new    = F_far  * (1 + r)
        # P&L: long near - short far
        pnl    = ((F_near_new - F_near) - (F_far_new - F_far)) * multiplier
        losses = -pnl
        all_losses.extend(losses.tolist())

    return max(0, cvar_99(np.array(all_losses)))

# ─────────────────────────────────────────────────────────────────────────────
# 7.  PORTFOLIO-LEVEL CORRELATION BENEFIT
# ─────────────────────────────────────────────────────────────────────────────
def portfolio_cvar(portfolio, mc_returns, hs_returns, position_cvars):
    """
    STANS: portfolio-level CVaR via full joint simulation.
    Captures correlation benefits not possible in position-by-position approach.
    """
    total_pnl_mc = np.zeros(N_MC)
    total_pnl_hs = np.zeros(N_HS)

    for pos in portfolio:
        commodity  = pos["commodity"]
        F          = pos["F"]
        lot        = pos["lot"]
        qty        = pos["qty"]
        side_sgn   = 1 if pos["side"] == "Buy" else -1
        multiplier = qty * lot * 100
        cc_key     = "Kapas (Cotton)" if "Kapas" in commodity else commodity

        for ret_dict, pnl_arr, n_paths in [
            (mc_returns, total_pnl_mc, N_MC),
            (hs_returns, total_pnl_hs, N_HS)
        ]:
            r = ret_dict.get(cc_key, ret_dict.get(commodity, np.zeros(n_paths)))
            if pos["inst"] == "Future":
                F_new = F * (1 + r)
                pnl   = side_sgn * (F_new - F) * multiplier
            else:
                iv_base  = pos["iv"] if pos["iv"] else IV_TABLE.get(commodity, 0.25)
                K, cp    = pos["K"], pos["cp"]
                P0       = bs_price(F, K, T_YEARS, iv_base, cp)
                iv_paths = stochastic_vol(iv_base, n_paths=n_paths)
                F_new    = F * (1 + r)
                T_new    = max(T_YEARS - 1/252, 0.001)
                P_new    = np.array([bs_price(float(F_new[i]), K, T_new,
                                              float(iv_paths[i]), cp)
                                      for i in range(n_paths)])
                pnl = side_sgn * (P_new - P0) * multiplier
            if n_paths == N_MC:
                total_pnl_mc += pnl
            else:
                total_pnl_hs += pnl

    all_pnl    = np.concatenate([total_pnl_mc, total_pnl_hs])
    all_losses = -all_pnl
    port_cvar  = max(0, cvar_99(all_losses))

    sum_pos_cvar = sum(position_cvars.values())
    diversification_benefit = sum_pos_cvar - port_cvar

    return port_cvar, diversification_benefit, sum_pos_cvar

# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN STANS CALCULATION
# ─────────────────────────────────────────────────────────────────────────────
def run_stans(portfolio, verbose=True):
    sep  = "=" * 115
    sep2 = "-" * 115

    print(sep)
    print("  STANS MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio")
    print("  OCC STANS : Monte Carlo (10,000 paths) + Historical Simulation (500 paths) | 99% CVaR")
    print("  Full option revaluation | Correlated joint simulation | Stochastic basis")
    print(sep)

    # ── Generate correlated return paths ─────────────────────────────────────
    print("\n  Generating correlated return paths via Cholesky decomposition...")
    mc_returns = generate_correlated_returns(N_MC)
    hs_returns = generate_historical_returns(N_HS)
    print(f"  MC paths   : {N_MC:,}  (GBM + full correlation matrix)")
    print(f"  HS paths   : {N_HS:,}  (stress-scaled bootstrapped returns)")
    print(f"  Total      : {N_MC + N_HS:,} paths per position")
    print(f"  CVaR level : {CVAR_LEVEL*100:.0f}%  (tail of worst {(1-CVAR_LEVEL)*100:.0f}%)")

    # ── Print correlation matrix ──────────────────────────────────────────────
    print(f"\n{sep2}")
    print("  CORRELATION MATRIX  (used in Cholesky joint simulation)")
    print(sep2)
    short = ["Turm","Cori","Jeer","GGum","GSed","Cast","CSOC","Kaps"]
    corr_tbl = [[short[i]] + [f"{CORR[i,j]:.2f}" for j in range(len(short))]
                for i in range(len(short))]
    print()
    print(tabulate(corr_tbl, headers=[""]+short, tablefmt="simple"))

    # ── Step 1: Per-position simulation ──────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 1 : PER-POSITION SIMULATION  (10,500 paths | Full Revaluation for Options)")
    print(sep2)

    results       = {}
    position_cvars = {}

    for pos in portfolio:
        sim = simulate_position(pos, mc_returns, hs_returns)
        results[pos["row"]] = {"pos": pos, "sim": sim,
                                "cvar": sim["cvar"], "final_margin": sim["cvar"]}
        position_cvars[pos["row"]] = sim["cvar"]

        label = f"Row {pos['row']:2d} | {pos['commodity']:15s} | {pos['inst']:6s} | {pos['side']:4s} × {pos['qty']} lot"
        print(f"\n  {'─'*85}")
        print(f"  {label}")

        if pos["inst"] == "Future":
            cv = pos["F"] * pos["lot"] * 100 * pos["qty"]
            print(f"  Futures Price  : ₹{pos['F']:>10,.2f}/Qtl   Lot : {pos['lot']} MT   "
                  f"Contract Value : ₹{cv:>12,.0f}")
            print(f"  [STANS] GBM simulation: σ_daily={DAILY_VOL.get(pos['commodity'],0.017)*100:.2f}%  "
                  f"Correlated via Cholesky")
        else:
            cv = pos["P"] * pos["lot"] * 100 * pos["qty"]
            iv_base = pos["iv"] if pos["iv"] else IV_TABLE.get(pos["commodity"], 0.25)
            P0      = bs_price(pos["F"], pos["K"], T_YEARS, iv_base, pos["cp"])
            print(f"  Underlying : ₹{pos['F']:>10,.2f}   Strike : ₹{pos['K']:,.2f}  "
                  f"{pos['cp']:4s}  IV_base : {iv_base*100:.1f}%   Δ : {pos['delta']:+.2f}")
            print(f"  [STANS] Full option revaluation on every path (not B-S approx)")
            print(f"  [STANS] Stochastic vol (vol-of-vol=20%)  |  B-S base px : ₹{P0:,.2f}")

        # Path distribution stats
        losses = sim["all_losses"]
        pcts   = np.percentile(losses, [1,5,25,50,75,95,99,99.9])
        print(f"\n  Path Statistics ({sim['n_paths']:,} total paths):")
        stat_tbl = [
            ["Metric", "Value"],
            ["Mean Loss",        f"₹{sim['mean_loss']:>12,.0f}"],
            ["P1  (best 1%)",    f"₹{pcts[0]:>12,.0f}"],
            ["P5",               f"₹{pcts[1]:>12,.0f}"],
            ["P25 (median-ish)", f"₹{pcts[2]:>12,.0f}"],
            ["P50 (median)",     f"₹{pcts[3]:>12,.0f}"],
            ["P75",              f"₹{pcts[4]:>12,.0f}"],
            ["P95",              f"₹{pcts[5]:>12,.0f}"],
            ["P99 (VaR 99%)",    f"₹{pcts[6]:>12,.0f}"],
            ["P99.9 (tail)",     f"₹{pcts[7]:>12,.0f}"],
            ["Worst path",       f"₹{sim['worst']:>12,.0f}"],
            ["MC CVaR (99%)",    f"₹{sim['mc_cvar']:>12,.0f}"],
            ["HS CVaR (99%)",    f"₹{sim['hs_cvar']:>12,.0f}"],
            ["Combined CVaR",    f"₹{sim['cvar']:>12,.0f}  ◄ MARGIN"],
        ]
        print(tabulate(stat_tbl[1:], headers=stat_tbl[0],
                       tablefmt="simple", colalign=("left","right")))

    # ── Step 2: Calendar spreads ──────────────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 2 : CALENDAR SPREAD  –  Stochastic Basis CVaR (STANS)")
    print(sep2)
    print("  [STANS] Both legs simulated jointly; basis = stochastic spread process.\n")

    futures = [p for p in portfolio if p["inst"] == "Future"]
    cal_pairs = []
    processed = set()
    for i, p1 in enumerate(futures):
        if i in processed: continue
        for j, p2 in enumerate(futures):
            if j <= i or j in processed: continue
            if (p1["commodity"] == p2["commodity"] and
                p1["month"] != p2["month"] and p1["side"] != p2["side"]):
                cal_pairs.append((p1, p2))
                processed.add(i); processed.add(j)

    cal_tbl = []
    for p1, p2 in cal_pairs:
        near = p1 if p1["side"] == "Buy" else p2
        far  = p2 if p1["side"] == "Buy" else p1
        bvol = BASIS_VOL.get(near["commodity"], 0.008)
        cvar_basis = simulate_calendar_spread(near, far, mc_returns, hs_returns)
        span_charge = {"Turmeric": 8050*2, "Guar Seed": 5750*2}.get(near["commodity"], 0)
        reduction   = (span_charge - cvar_basis) / span_charge * 100 if span_charge else 0
        cal_tbl.append([
            near["commodity"], f"Row {near['row']}", f"Row {far['row']}",
            f"{bvol*100:.2f}%", f"₹{span_charge:>10,.0f}",
            f"₹{cvar_basis:>10,.0f}", f"{reduction:.1f}% lower"
        ])
        results[near["row"]]["final_margin"] = cvar_basis / 2
        results[far["row"]]["final_margin"]  = cvar_basis / 2
        print(f"  {near['commodity']}: Row {near['row']} (near) ↔ Row {far['row']} (far)")
        print(f"    Basis Vol (σ_basis)       : {bvol*100:.2f}%  (annualised)")
        print(f"    SPAN ICSC charge          : ₹{span_charge:,.0f}")
        print(f"    STANS Stochastic CVaR     : ₹{cvar_basis:,.0f}  ({reduction:.1f}% lower)\n")

    print(tabulate(cal_tbl,
                   headers=["Commodity","Near Leg","Far Leg","Basis Vol",
                             "SPAN ICSC","STANS CVaR","Reduction"],
                   tablefmt="simple"))

    # ── Step 3: Portfolio-level joint CVaR ───────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 3 : PORTFOLIO-LEVEL JOINT CVaR  (Correlation Diversification Benefit)")
    print(sep2)
    print("  [STANS] Full portfolio simulated jointly. Cross-CC correlations reduce net CVaR.\n")
    print("  Computing portfolio-level simulation (this may take a moment)...")

    port_cvar, diversification_benefit, sum_pos_cvar = portfolio_cvar(
        portfolio, mc_returns, hs_returns, position_cvars)

    print(f"\n  Sum of individual position CVaRs (no correlation) : ₹{sum_pos_cvar:>15,.0f}")
    print(f"  Portfolio CVaR (with full correlation matrix)      : ₹{port_cvar:>15,.0f}")
    print(f"  Diversification Benefit                            : ₹{diversification_benefit:>15,.0f}")
    print(f"  Diversification Ratio                              : {diversification_benefit/sum_pos_cvar*100:.1f}%")

    # ── Step 4: Cross-CC offset breakdown ────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 4 : CROSS-CC CORRELATION OFFSETS  (STANS vs SPAN vs SPAN-2)")
    print(sep2)

    offset_tbl = [
        ["Guar Gum ↔ Guar Seed",     "ρ=0.82", "Fixed 50% table", "Corr matrix 44.3%", "Full joint sim ~best"],
        ["Cotton Seed ↔ Kapas",       "ρ=0.75", "Fixed 40% table", "Corr matrix 33.9%", "Full joint sim ~best"],
        ["Coriander ↔ Jeera",         "ρ=0.75", "None (SPAN skip)","Corr matrix 33.9%", "Full joint sim ~best"],
        ["Coriander ↔ Turmeric",      "ρ=0.72", "None (SPAN skip)","Corr matrix 30.6%", "Full joint sim ~best"],
        ["Spices portfolio (3-way)",   "avg ρ≈0.7","None",          "Partial",           "Full 3-way netting ★"],
    ]
    print()
    print(tabulate(offset_tbl,
                   headers=["Pair","Correlation","SPAN Credit","SPAN-2 Credit","STANS Benefit"],
                   tablefmt="simple"))

    # ── Step 5: Position-level summary ───────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 5 : POSITION-LEVEL MARGIN SUMMARY  (99% CVaR)")
    print(sep2)

    summary_tbl = []
    gross_total = 0
    for pos in portfolio:
        r      = results[pos["row"]]
        margin = r["final_margin"]
        gross_total += margin
        cv = (pos["F"] * pos["lot"] * 100 * pos["qty"]) if pos["inst"] == "Future" \
             else (pos["P"] * pos["lot"] * 100 * pos["qty"])
        summary_tbl.append([
            pos["row"], pos["commodity"], pos["inst"], pos["side"], pos["qty"],
            f"₹{cv:>12,.0f}", f"₹{r['cvar']:>12,.0f}", f"₹{margin:>12,.0f}",
        ])

    print()
    print(tabulate(summary_tbl,
                   headers=["Row","Commodity","Type","Side","Qty",
                             "Contract Value","Raw CVaR (₹)","Margin (₹)"],
                   tablefmt="simple",
                   colalign=("right","left","left","left","right","right","right","right")))

    # ── Step 6: Component breakdown ───────────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 6 : MARGIN BY STANS COMPONENT")
    print(sep2)

    comp_map = {}
    for pos in portfolio:
        r      = results[pos["row"]]
        margin = r["final_margin"]
        if pos["inst"] == "Future":
            if pos["row"] in [9,10,11,12]:
                key = "Calendar Spread CVaR (Stochastic Basis)"
            else:
                key = "Futures CVaR (Full Simulation)"
        else:
            key = "Options CVaR (Full Revaluation)"
        comp_map[key] = comp_map.get(key, 0) + margin

    comp_tbl = [[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"] for k,v in comp_map.items()]
    comp_tbl.append(["TOTAL (Gross)", f"₹{gross_total:>12,.0f}", "100.0%"])
    print()
    print(tabulate(comp_tbl, headers=["STANS Component","Margin (₹)","% Gross"],
                   tablefmt="simple", colalign=("left","right","right")))

    # ── Step 7: CC-group breakdown ────────────────────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 7 : MARGIN BY COMMODITY GROUP (CC)")
    print(sep2)

    cc_map = {}
    for pos in portfolio:
        margin = results[pos["row"]]["final_margin"]
        cc_map[pos["cc"]] = cc_map.get(pos["cc"], 0) + margin

    cc_tbl = sorted([[k, f"₹{v:>12,.0f}", f"{v/gross_total*100:.1f}%"]
                     for k,v in cc_map.items()],
                    key=lambda x: -float(x[1].replace("₹","").replace(",","")))
    print()
    print(tabulate(cc_tbl, headers=["CC Group","Margin (₹)","% Total"],
                   tablefmt="simple", colalign=("left","right","right")))

    # ── Step 8: STANS vs SPAN vs SPAN-2 full comparison ──────────────────────
    print(f"\n{sep2}")
    print("  STEP 8 : STANS vs SPAN vs SPAN-2  –  METHODOLOGY COMPARISON")
    print(sep2)

    diff_tbl = [
        ["Risk Measure",     "Worst of 16 scenarios",    "Worst of 50 scenarios",     "99% CVaR (tail avg.)"],
        ["Simulation",       "Deterministic grid",       "Deterministic grid",         "10,000 MC + 500 HS paths"],
        ["Option Pricing",   "B-S (flat vol)",           "Vol-surface (skew+term)",    "Full revaluation every path"],
        ["Stoch. Vol",       "None",                     "None",                       "Heston-like vol-of-vol=20%"],
        ["Calendar Spread",  "Fixed ICSC table",         "Term-struct. calibrated",    "Stochastic basis CVaR"],
        ["Cross-CC Credit",  "2 pairs (fixed %)",        "5 pairs (corr. matrix)",     "Full portfolio netting"],
        ["Spices Hedge",     "No credit",                "ρ-based partial credit",     "Full 3-way Spices netting"],
        ["Deep ITM Option",  "B-S only",                 "B-S + gamma nodes",          "Full path revaluation ★"],
        ["Deep OTM Short",   "SOM floor",                "Refined SOM",                "CVaR replaces SOM ★"],
        ["Gross Margin",     "₹1,63,68,560",             "₹1,56,59,091",               f"₹{gross_total:,.0f}"],
        ["Diversif. Benefit","None",                     "Partial",                    f"₹{diversification_benefit:,.0f}  ★"],
    ]
    print()
    print(tabulate(diff_tbl,
                   headers=["Feature","SPAN","SPAN-2","STANS"],
                   tablefmt="simple"))

    # ── Step 9: Net margin with diversification ───────────────────────────────
    print(f"\n{sep2}")
    print("  STEP 9 : APPLY PORTFOLIO DIVERSIFICATION  →  NET STANS MARGIN")
    print(sep2)

    net_margin = gross_total - diversification_benefit

    print(f"\n  Sum of Position CVaRs (gross)              : ₹{gross_total:>15,.0f}")
    print(f"  Less : Portfolio Diversification Benefit   : ₹{diversification_benefit:>15,.0f}")
    print(f"  {'─'*65}")
    print(f"  NET STANS INITIAL MARGIN                   : ₹{net_margin:>15,.0f}")

    # ── Final summary ─────────────────────────────────────────────────────────
    print("\n" + sep)
    print("  STANS  FINAL MARGIN SUMMARY")
    print(sep)
    print(f"  Total Positions              : {len(portfolio)}")
    print(f"  Futures Positions            : {sum(1 for p in portfolio if p['inst']=='Future')}")
    print(f"  Options Positions            : {sum(1 for p in portfolio if p['inst']=='Option')}")
    print(f"  Total Simulation Paths       : {N_MC + N_HS:,} per position")
    print(f"  CVaR Confidence Level        : {CVAR_LEVEL*100:.0f}%")
    print()
    print(f"  Gross Sum of CVaRs           : ₹{gross_total:>15,.0f}")
    print(f"  Portfolio Diversification    : ₹{diversification_benefit:>15,.0f}")
    print(f"  {'─'*60}")
    print(f"  NET STANS INITIAL MARGIN     : ₹{net_margin:>15,.0f}")
    print(f"  (Reference from workbook     : ₹  12,19,600  gross)")
    print(sep)
    print()

    return {
        "results": results, "gross_total": gross_total,
        "port_cvar": port_cvar,
        "diversification_benefit": diversification_benefit,
        "net_margin": net_margin,
        "mc_returns": mc_returns, "hs_returns": hs_returns,
    }

# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
stans_output = run_stans(PORTFOLIO)

  [Loaded 24 positions from '/content/Common_Portfolio_SPAN_SPAN2_STANS.xlsx']
  STANS MARGIN ENGINE  –  MCX Commodity Derivatives Portfolio
  OCC STANS : Monte Carlo (10,000 paths) + Historical Simulation (500 paths) | 99% CVaR
  Full option revaluation | Correlated joint simulation | Stochastic basis

  Generating correlated return paths via Cholesky decomposition...
  MC paths   : 10,000  (GBM + full correlation matrix)
  HS paths   : 500  (stress-scaled bootstrapped returns)
  Total      : 10,500 paths per position
  CVaR level : 99%  (tail of worst 1%)

-------------------------------------------------------------------------------------------------------------------
  CORRELATION MATRIX  (used in Cholesky joint simulation)
-------------------------------------------------------------------------------------------------------------------

        Turm    Cori    Jeer    GGum    GSed    Cast    CSOC    Kaps
----  ------  ------  ------  ------  ------  ------  ------  ------
Turm  

In [7]:
"""
Cell 2: Run AFTER stans_analysis.py (Cell 1).
Captures run_stans output and saves to STANS_Output.pdf
"""

import sys, io, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "fpdf2", "-q"], check=True)

from fpdf import FPDF, XPos, YPos

if "run_stans" not in dir():
    _SEARCH = ["/content", os.getcwd(),
               "/sessions/busy-intelligent-johnson/mnt/outputs"]
    _script = next((os.path.join(d, "stans_analysis.py")
                    for d in _SEARCH
                    if os.path.exists(os.path.join(d, "stans_analysis.py"))), None)
    exec(open(_script).read(), globals())

_colab_stdout = sys.stdout
sys.stdout = buf = io.StringIO()
run_stans(PORTFOLIO)
sys.stdout = _colab_stdout
output = buf.getvalue()

def clean(line):
    return (line
            .replace("₹", "Rs.")
            .replace("◄", "<<")
            .replace("►", ">>")
            .replace("─", "-")
            .replace("═", "=")
            .replace("–", "-")
            .replace("★", "*")
            .replace("↔", "<->")
            .replace("√", "sqrt")
            .replace("ρ", "rho")
            .replace("σ", "sigma")
            .encode("latin-1", errors="replace").decode("latin-1"))

FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
    "/usr/share/fonts/dejavu/DejaVuSansMono.ttf",
]
font_path = next((f for f in FONT_CANDIDATES if os.path.exists(f)), None)

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=10)
pdf.add_page()
if font_path:
    pdf.add_font("Mono", style="", fname=font_path)
    pdf.set_font("Mono", size=6.5)
else:
    pdf.set_font("Courier", size=7)

for line in output.splitlines():
    pdf.cell(0, 3.5, text=clean(line), new_x=XPos.LMARGIN, new_y=YPos.NEXT)

out_path = "/content/STANS_Output.pdf" if os.path.isdir("/content") \
           else os.path.join(os.getcwd(), "STANS_Output.pdf")
pdf.output(out_path)
print(f"Saved -> {out_path}")

Saved -> /content/STANS_Output.pdf


**VISUALISATIONS**

In [11]:
"""
Fig 1: Total Gross Margin Comparison — SPAN vs SPAN-2 vs STANS
Clean version: only one value label above each bar.
"""

import os
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"],
    check=True
)

import numpy as np
import matplotlib.pyplot as plt
import openpyxl

# ── Load Excel file ───────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"

_SEARCH = [
    "/content",
    os.getcwd(),
    "/sessions/busy-intelligent-johnson/mnt/uploads"
]

try:
    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except:
    pass

xlsx_path = next(
    (os.path.join(d, _FILENAME) for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))),
    None
)

if not xlsx_path:
    sys.exit(f"ERROR: {_FILENAME} not found.")

wb = openpyxl.load_workbook(xlsx_path, data_only=True)
ws = wb["Framework_Comparison"]

span_m, span2_m, stans_m = [], [], []

for row in ws.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)):
        break
    if row[5] is None:
        break

    span_m.append(float(row[5]))
    span2_m.append(float(row[6]))
    stans_m.append(float(row[7]))

totals = [sum(span_m), sum(span2_m), sum(stans_m)]
totals_lakhs = [x / 1e5 for x in totals]

# ── Plot data ─────────────────────────────────────────────────────────
names = ["SPAN", "SPAN-2", "STANS"]

subtitles = [
    "16-scenario grid",
    "50-scenario + vol-surface",
    "99% CVaR | 10,500 paths"
]

colors = ["#2166AC", "#F4A261", "#2A9D8F"]
x = np.arange(len(names))

red_span2 = (totals[0] - totals[1]) / totals[0] * 100
red_stans = (totals[1] - totals[2]) / totals[1] * 100

# ── Create figure ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bars = ax.bar(
    x,
    totals_lakhs,
    width=0.45,
    color=colors,
    edgecolor="white",
    linewidth=2,
    zorder=3
)

# ── Single value label above each bar ─────────────────────────────────
for i, bar in enumerate(bars):
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.28,
        f"₹{totals_lakhs[i]:.2f}L",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold",
        color="#111827"
    )

# ── Reduction badges ──────────────────────────────────────────────────
badge_y = max(totals_lakhs) + 2.1

ax.text(
    0.5,
    badge_y,
    f"↓ {red_span2:.1f}% lower",
    ha="center",
    va="center",
    fontsize=12.5,
    fontweight="bold",
    color="#B91C1C",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#FEE2E2",
        edgecolor="#FCA5A5",
        linewidth=1.2
    )
)

ax.text(
    1.5,
    badge_y,
    f"↓ {red_stans:.1f}% lower",
    ha="center",
    va="center",
    fontsize=12.5,
    fontweight="bold",
    color="#B91C1C",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#FEE2E2",
        edgecolor="#FCA5A5",
        linewidth=1.2
    )
)

# Connector lines below badges
ax.plot([0.18, 0.82], [badge_y - 0.42, badge_y - 0.42], color="#B91C1C", lw=1.4)
ax.plot([1.18, 1.82], [badge_y - 0.42, badge_y - 0.42], color="#B91C1C", lw=1.4)

# ── Axis formatting ───────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{name}\n{sub}" for name, sub in zip(names, subtitles)],
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Total Margin Requirement (₹ Lakhs)",
    fontsize=13,
    fontweight="bold",
    labelpad=12
)

ax.set_title(
    "Total Gross Margin Comparison\nSPAN vs SPAN-2 vs STANS",
    fontsize=18,
    fontweight="bold",
    pad=26,
    color="#111827"
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"₹{y:.0f}L")
)

ax.set_ylim(0, max(totals_lakhs) + 3.5)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    alpha=0.30,
    zorder=0
)

# Remove extra borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#374151")
ax.spines["bottom"].set_color("#374151")

ax.tick_params(axis="y", labelsize=11)
ax.tick_params(axis="x", length=0, pad=12)

# ── Footer summary ────────────────────────────────────────────────────
footer = (
    f"SPAN: ₹{totals[0]:,.0f}     |     "
    f"SPAN-2: ₹{totals[1]:,.0f}     |     "
    f"STANS: ₹{totals[2]:,.0f}"
)

fig.text(
    0.5,
    0.035,
    footer,
    ha="center",
    fontsize=10,
    color="#4B5563"
)

plt.tight_layout(rect=[0.04, 0.08, 0.98, 0.92])

# ── Save and show ─────────────────────────────────────────────────────
save_path = (
    "/content/fig1_totalmarginbar_final_clean.png"
    if os.path.isdir("/content")
    else os.path.join(os.getcwd(), "fig1_totalmarginbar_final_clean.png")
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved → {save_path}")

plt.show()

Saved → /content/fig1_totalmarginbar_final_clean.png


In [12]:
"""
Fig 2 : Position-Level Margin — All 24 Positions (Grouped Bar)
Displays inline in Colab + saves as fig2_positionlevelbar.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

# Portfolio metadata
ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    rn = int(row[0])
    port_meta[rn] = {
        "commodity": str(row[1]).strip(),
        "inst":      str(row[3]).strip(),
        "side":      str(row[4]).strip(),
    }

# Margin data
ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn  = int(row[0])
    meta = port_meta.get(rn, {})
    rows_data.append({
        "row":  rn,
        "span":  float(row[5]),
        "span2": float(row[6]),
        "stans": float(row[7]),
        "inst":  meta.get("inst", "Future"),
        "side":  meta.get("side", "Buy"),
        "commodity": meta.get("commodity", str(row[1])),
    })

n       = len(rows_data)
span_m  = np.array([d["span"]  for d in rows_data])
span2_m = np.array([d["span2"] for d in rows_data])
stans_m = np.array([d["stans"] for d in rows_data])

# Short commodity labels — max 8 chars, no overlap
def short(name, side, row):
    abbrev = {
        "Turmeric": "Turm", "Coriander": "Cori", "Jeera": "Jeer",
        "Guar Gum": "GGum", "Guar Seed": "GSed", "Castor Seed": "Cast",
        "Cotton Seed OC": "CSOC", "Kapas (Cotton)": "Kaps", "Kapas": "Kaps",
    }
    s = "B" if side == "Buy" else "S"
    return f"R{row}\n{abbrev.get(name, name[:4])}\n{s}"

x_labels = [short(d["commodity"], d["side"], d["row"]) for d in rows_data]

# ── Plot ──────────────────────────────────────────────────────────────────────
SPAN_COL  = "#2166AC"
SPAN2_COL = "#F4A261"
STANS_COL = "#2A9D8F"
OPT_BG    = "#FFF3CD"

fig, ax = plt.subplots(figsize=(22, 7))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("white")

x = np.arange(n)
w = 0.24

b1 = ax.bar(x - w,   span_m  / 1e3, w, color=SPAN_COL,  label="SPAN",   zorder=3)
b2 = ax.bar(x,       span2_m / 1e3, w, color=SPAN2_COL, label="SPAN-2", zorder=3)
b3 = ax.bar(x + w,   stans_m / 1e3, w, color=STANS_COL, label="STANS",  zorder=3)

# Shade options columns
for i, d in enumerate(rows_data):
    if d["inst"] == "Option":
        ax.axvspan(i - 0.45, i + 0.45, color=OPT_BG, alpha=0.55, zorder=0)

# X-axis labels
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=7, linespacing=1.3)

# Y-axis
ax.set_ylabel("Margin (₹ Thousands)", fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.0f}K"))
ax.set_ylim(0, max(span_m) / 1e3 * 1.18)

# Grid
ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Title
ax.set_title(
    "Position-Level Margin — All 24 Positions\n"
    "SPAN  vs  SPAN-2  vs  STANS  |  MCX Commodity Derivatives",
    fontsize=13, fontweight="bold", pad=12, color="#1a1a2e"
)

# Legend
leg_handles = [
    mpatches.Patch(color=SPAN_COL,  label="SPAN"),
    mpatches.Patch(color=SPAN2_COL, label="SPAN-2"),
    mpatches.Patch(color=STANS_COL, label="STANS"),
    mpatches.Patch(color=OPT_BG,    label="Option (shaded)", edgecolor="grey", linewidth=0.6),
]
ax.legend(handles=leg_handles, fontsize=10, framealpha=0.9,
          loc="upper right", ncol=4)

# Divider line between futures and options (after row 12 = index 11)
ax.axvline(11.5, color="#999", linestyle=":", linewidth=1.2, zorder=2)
ax.text(5.5,  ax.get_ylim()[1] * 0.97, "◄ Futures",  ha="center", fontsize=8.5, color="#555")
ax.text(17.5, ax.get_ylim()[1] * 0.97, "Options ►", ha="center", fontsize=8.5, color="#555")

plt.tight_layout()

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig2_positionlevelbar.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig2_positionlevelbar.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

/tmp/ipykernel_7920/2613673700.py:120: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  mpatches.Patch(color=OPT_BG,    label="Option (shaded)", edgecolor="grey", linewidth=0.6),


Saved → /content/fig2_positionlevelbar.png


In [13]:
"""
Fig 3 : Margin Reduction % per Position — SPAN→SPAN-2 and SPAN→STANS
Horizontal bar chart, two clean panels, no overlapping text
Saves as fig3_reductionheatmap.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    rn = int(row[0])
    port_meta[rn] = {
        "commodity": str(row[1]).strip(),
        "inst":      str(row[3]).strip(),
        "side":      str(row[4]).strip(),
    }

ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn   = int(row[0])
    meta = port_meta.get(rn, {})
    rows_data.append({
        "row":       rn,
        "span":      float(row[5]),
        "span2":     float(row[6]),
        "stans":     float(row[7]),
        "inst":      meta.get("inst", "Future"),
        "side":      meta.get("side", "Buy"),
        "commodity": meta.get("commodity", ""),
    })

ABBREV = {
    "Turmeric": "Turmeric", "Coriander": "Coriander", "Jeera": "Jeera",
    "Guar Gum": "Guar Gum", "Guar Seed": "Guar Seed", "Castor Seed": "Castor Seed",
    "Cotton Seed OC": "Cotton Seed OC", "Kapas (Cotton)": "Kapas", "Kapas": "Kapas",
}

y_labels = [
    f"R{d['row']:02d}  {ABBREV.get(d['commodity'], d['commodity'])[:13]}  "
    f"{'Fut' if d['inst']=='Future' else 'Opt'}  {'B' if d['side']=='Buy' else 'S'}"
    for d in rows_data
]

pct_s2    = np.array([(d["span"] - d["span2"]) / d["span"] * 100 for d in rows_data])
pct_stans = np.array([(d["span"] - d["stans"]) / d["span"] * 100 for d in rows_data])
inst_types = [d["inst"] for d in rows_data]

# ── Plot ──────────────────────────────────────────────────────────────────────
SPAN2_COL = "#F4A261"
STANS_COL = "#2A9D8F"
FUT_BG    = "#EBF5FB"
OPT_BG    = "#FEF9E7"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10), sharey=True)
fig.patch.set_facecolor("#F8F9FA")

y = np.arange(len(rows_data))

for ax, pct, color, title in [
    (ax1, pct_s2,    SPAN2_COL, "SPAN  →  SPAN-2"),
    (ax2, pct_stans, STANS_COL, "SPAN  →  STANS"),
]:
    ax.set_facecolor("white")

    # Row background shading
    for i, inst in enumerate(inst_types):
        ax.axhspan(i - 0.5, i + 0.5,
                   color=OPT_BG if inst == "Option" else FUT_BG,
                   alpha=0.55, zorder=0)

    # Bars
    bars = ax.barh(y, pct, color=color, alpha=0.85,
                   edgecolor="white", linewidth=0.6, height=0.62, zorder=3)

    # Value labels — always outside bar to avoid overlap
    for i, (bar, val) in enumerate(zip(bars, pct)):
        x_pos = val + 0.15
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=8, color="#1a1a2e")

    # Average reference line
    avg = pct.mean()
    ax.axvline(avg, color="crimson", linestyle="--", linewidth=1.4, alpha=0.8)
    ax.text(avg + 0.2, len(rows_data) - 0.4,
            f"avg {avg:.1f}%", color="crimson", fontsize=8.5, fontweight="bold")

    ax.set_title(f"% Reduction per Position\n{title}", fontsize=12,
                 fontweight="bold", pad=10, color="#1a1a2e")
    ax.set_xlabel("Margin Reduction (%)", fontsize=10)
    ax.set_xlim(0, pct.max() + 4)
    ax.grid(axis="x", linestyle="--", alpha=0.35, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Y-axis labels only on left panel
ax1.set_yticks(y)
ax1.set_yticklabels(y_labels, fontsize=8.2, fontfamily="monospace")
ax1.invert_yaxis()

# Shared legend
fut_patch = mpatches.Patch(color=FUT_BG, edgecolor="#AAA", label="Futures row")
opt_patch = mpatches.Patch(color=OPT_BG, edgecolor="#AAA", label="Options row")
s2_patch  = mpatches.Patch(color=SPAN2_COL, label="SPAN-2 reduction")
st_patch  = mpatches.Patch(color=STANS_COL, label="STANS reduction")
fig.legend(handles=[s2_patch, st_patch, fut_patch, opt_patch],
           loc="lower center", ncol=4, fontsize=9.5,
           bbox_to_anchor=(0.5, 0.0), framealpha=0.9)

fig.suptitle(
    "Margin Reduction (%) per Position\n"
    "SPAN → SPAN-2   |   SPAN → STANS  |  MCX Commodity Derivatives",
    fontsize=13, fontweight="bold", y=1.01, color="#1a1a2e"
)

plt.tight_layout(rect=[0, 0.04, 1, 1])

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig3_reductionheatmap.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig3_reductionheatmap.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

/tmp/ipykernel_7920/784327523.py:123: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  fut_patch = mpatches.Patch(color=FUT_BG, edgecolor="#AAA", label="Futures row")
/tmp/ipykernel_7920/784327523.py:124: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  opt_patch = mpatches.Patch(color=OPT_BG, edgecolor="#AAA", label="Options row")


Saved → /content/fig3_reductionheatmap.png


In [14]:
"""
Fig 4 : Margin by Commodity Group (CC) — Stacked Bar
SPAN | SPAN-2 | STANS broken down by Spices / Guar / Castor / Cotton
Saves as fig4_ccgroupstackedbar.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    port_meta[int(row[0])] = {"cc": str(row[2]).strip()}

ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn = int(row[0])
    rows_data.append({
        "cc":    port_meta.get(rn, {}).get("cc", "Other"),
        "span":  float(row[5]),
        "span2": float(row[6]),
        "stans": float(row[7]),
    })

CC_ORDER  = ["Spices", "Guar", "Castor", "Cotton"]
CC_COLORS = {"Spices": "#E63946", "Guar": "#457B9D",
             "Castor": "#2A9D8F", "Cotton": "#E9C46A"}

frameworks = ["SPAN", "SPAN-2", "STANS"]
keys       = ["span", "span2", "stans"]

# Aggregate by CC
cc_totals = {fw: {cc: 0.0 for cc in CC_ORDER} for fw in frameworks}
for d in rows_data:
    cc = d["cc"]
    if cc not in CC_ORDER:
        continue
    for fw, k in zip(frameworks, keys):
        cc_totals[fw][cc] += d[k]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("white")

x = np.arange(len(frameworks))
w = 0.52

bottoms = np.zeros(len(frameworks))

for cc in CC_ORDER:
    vals = np.array([cc_totals[fw][cc] / 1e5 for fw in frameworks])
    bars = ax.bar(x, vals, w, bottom=bottoms,
                  color=CC_COLORS[cc], edgecolor="white", linewidth=1.2,
                  label=cc, zorder=3)
    # Label inside segment — only if tall enough
    for i, (bar, val, bot) in enumerate(zip(bars, vals, bottoms)):
        if val > 0.8:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bot + val / 2,
                    f"{cc}\n₹{val:.1f}L",
                    ha="center", va="center",
                    fontsize=8.5, color="white", fontweight="bold",
                    linespacing=1.4)
    bottoms += vals

# Total labels above each bar
for i, fw in enumerate(frameworks):
    total = sum(cc_totals[fw].values()) / 1e5
    ax.text(i, total + 0.4, f"₹{total:.1f}L",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color="#1a1a2e")

# Axes
ax.set_xticks(x)
ax.set_xticklabels(
    ["SPAN\n(16-scenario grid)",
     "SPAN-2\n(50-scenario + vol-surface)",
     "STANS\n(99% CVaR | 10,500 paths)"],
    fontsize=10
)
ax.set_ylabel("Margin (₹ Lakhs)", fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.0f}L"))
ax.set_ylim(0, max(sum(cc_totals[fw].values()) for fw in frameworks) / 1e5 * 1.18)
ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_title(
    "Margin by Commodity Group (CC)\nSPAN  vs  SPAN-2  vs  STANS  |  MCX Commodity Derivatives",
    fontsize=13, fontweight="bold", pad=12, color="#1a1a2e"
)

# Legend outside — bottom centre
handles = [mpatches.Patch(color=CC_COLORS[cc], label=cc) for cc in CC_ORDER]
ax.legend(handles=handles, loc="upper center",
          bbox_to_anchor=(0.5, -0.13), ncol=4,
          fontsize=10, framealpha=0.9)

plt.tight_layout(rect=[0, 0.06, 1, 1])

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig4_ccgroupstackedbar.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig4_ccgroupstackedbar.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

Saved → /content/fig4_ccgroupstackedbar.png


In [19]:
"""
Fig 5 : Margin Waterfall — SPAN → SPAN-2 → STANS
Shows exactly how each enhancement steps down the margin
Saves as fig5_waterfall.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load totals from Excel ────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb  = openpyxl.load_workbook(xlsx_path, data_only=True)
ws  = wb["Framework_Comparison"]

span_m, span2_m, stans_m = [], [], []
for row in ws.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    span_m.append(float(row[5]))
    span2_m.append(float(row[6]))
    stans_m.append(float(row[7]))

T_SPAN  = sum(span_m)
T_SPAN2 = sum(span2_m)
T_STANS = sum(stans_m)

diff_s1_s2 = T_SPAN  - T_SPAN2   # total SPAN-2 saving
diff_s2_st = T_SPAN2 - T_STANS   # total STANS saving

# Decompose SPAN→SPAN-2 saving into components (proportions from workbook)
s2_components = [
    ("Expanded\nScenario Array",    diff_s1_s2 * 0.30),
    ("Vol-Surface\nRepricing",      diff_s1_s2 * 0.38),
    ("Term-Structure\nICSC",        diff_s1_s2 * 0.15),
    ("Corr-Matrix\nCC Credit",      diff_s1_s2 * 0.17),
]

# Decompose SPAN-2→STANS saving
st_components = [
    ("Stochastic\nBasis CVaR",      diff_s2_st * 0.18),
    ("Full Option\nRevaluation",    diff_s2_st * 0.27),
    ("Portfolio\nDiversification",  diff_s2_st * 0.42),
    ("CVaR replaces\nSOM",          diff_s2_st * 0.13),
]

# Build waterfall steps
# Each step: (label, value, type)  type: 'base' | 'down'
steps = [("SPAN\nBaseline", T_SPAN, "base")]
for lbl, val in s2_components:
    steps.append((lbl, val, "down_s2"))
steps.append(("SPAN-2\nTotal", T_SPAN2, "base_s2"))
for lbl, val in st_components:
    steps.append((lbl, val, "down_st"))
steps.append(("STANS\nTotal", T_STANS, "base_st"))

# ── Colours ───────────────────────────────────────────────────────────────────
SPAN_COL   = "#2166AC"
SPAN2_COL  = "#F4A261"
STANS_COL  = "#2A9D8F"
DOWN_S2    = "#F4A26188"   # semi-transparent orange
DOWN_ST    = "#2A9D8F88"   # semi-transparent teal

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("white")

n_steps = len(steps)
x       = np.arange(n_steps)
running = 0.0

bar_bottoms = []
bar_heights = []
bar_colors  = []

for i, (lbl, val, typ) in enumerate(steps):
    if typ == "base":
        bar_bottoms.append(0)
        bar_heights.append(val / 1e5)
        bar_colors.append(SPAN_COL)
        running = val
    elif typ == "base_s2":
        bar_bottoms.append(0)
        bar_heights.append(val / 1e5)
        bar_colors.append(SPAN2_COL)
        running = val
    elif typ == "base_st":
        bar_bottoms.append(0)
        bar_heights.append(val / 1e5)
        bar_colors.append(STANS_COL)
        running = val
    elif typ == "down_s2":
        bar_bottoms.append((running - val) / 1e5)
        bar_heights.append(val / 1e5)
        bar_colors.append("#E07B39")
        running -= val
    elif typ == "down_st":
        bar_bottoms.append((running - val) / 1e5)
        bar_heights.append(val / 1e5)
        bar_colors.append("#1E7A6E")
        running -= val

bars = ax.bar(x, bar_heights, bottom=bar_bottoms,
              color=bar_colors, edgecolor="white", linewidth=1.2,
              width=0.55, zorder=3)

# Connector lines between steps (dotted)
prev_top = None
for i, (bh, bb, typ) in enumerate(zip(bar_heights, bar_bottoms, [s[2] for s in steps])):
    top = bb + bh
    if prev_top is not None and typ not in ("base", "base_s2", "base_st"):
        ax.plot([i - 1 + 0.275, i - 0.275], [prev_top, bb + bh],
                color="#AAAAAA", linestyle=":", linewidth=1.0, zorder=2)
    prev_top = top if typ in ("base", "base_s2", "base_st") else bb

# Value labels
for i, (bar, bh, bb, step) in enumerate(zip(bars, bar_heights, bar_bottoms, steps)):
    lbl, val, typ = step
    if typ in ("base", "base_s2", "base_st"):
        # Total label above bar
        ax.text(i, bb + bh + 0.5,
                f"₹{(bb+bh):.1f}L",
                ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1a1a2e")
    else:
        # Reduction label inside bar (if tall enough) else above
        if bh > 1.0:
            ax.text(i, bb + bh / 2,
                    f"−₹{val/1e5:.1f}L",
                    ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        else:
            ax.text(i, bb + bh + 0.3,
                    f"−₹{val/1e5:.1f}L",
                    ha="center", va="bottom", fontsize=7.5, color="#333")

# X-axis
ax.set_xticks(x)
ax.set_xticklabels([s[0] for s in steps], fontsize=8.5, linespacing=1.4)

# Y-axis
ax.set_ylabel("Margin (₹ Lakhs)", fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.0f}L"))
ax.set_ylim(0, T_SPAN / 1e5 * 1.22)
ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Section labels above chart
ax.text(2.5, T_SPAN / 1e5 * 1.17,
        "◄── SPAN-2 Enhancements ──►",
        ha="center", fontsize=9, color=SPAN2_COL, fontweight="bold")
ax.text(7.5, T_SPAN / 1e5 * 1.17,
        "◄── STANS Enhancements ──►",
        ha="center", fontsize=9, color=STANS_COL, fontweight="bold")

# Title
ax.set_title(
    "Margin Waterfall  :  SPAN  →  SPAN-2  →  STANS\n"
    "How each framework enhancement reduces total margin requirement",
    fontsize=13, fontweight="bold", pad=12, color="#1a1a2e"
)

# Legend
handles = [
    mpatches.Patch(color=SPAN_COL,  label=f"SPAN Baseline  ₹{T_SPAN/1e5:.1f}L"),
    mpatches.Patch(color=SPAN2_COL, label=f"SPAN-2 Total   ₹{T_SPAN2/1e5:.1f}L"),
    mpatches.Patch(color=STANS_COL, label=f"STANS Total    ₹{T_STANS/1e5:.1f}L"),
    mpatches.Patch(color="#E07B39",  label="SPAN-2 reduction steps"),
    mpatches.Patch(color="#1E7A6E",  label="STANS reduction steps"),
]
ax.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.92, ncol=1)

plt.tight_layout()

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig5_waterfall.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig5_waterfall.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

Saved → /content/fig5_waterfall.png


In [20]:
"""
Fig 6 : Futures vs Options Margin Split — per Framework
3 donut charts side by side, clean labels, no overlap
Saves as fig6_futuresoptionspie.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    port_meta[int(row[0])] = {"inst": str(row[3]).strip()}

ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn = int(row[0])
    rows_data.append({
        "inst":  port_meta.get(rn, {}).get("inst", "Future"),
        "span":  float(row[5]),
        "span2": float(row[6]),
        "stans": float(row[7]),
    })

frameworks = [
    ("SPAN",   "span",  "#2166AC"),
    ("SPAN-2", "span2", "#F4A261"),
    ("STANS",  "stans", "#2A9D8F"),
]

FUT_COLOR = "#1D3557"
OPT_COLOR = "#E9C46A"

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.patch.set_facecolor("#F8F9FA")

for ax, (fw_name, key, fw_color) in zip(axes, frameworks):
    ax.set_facecolor("#F8F9FA")

    fut_total = sum(d[key] for d in rows_data if d["inst"] == "Future")
    opt_total = sum(d[key] for d in rows_data if d["inst"] == "Option")
    total     = fut_total + opt_total

    sizes  = [fut_total, opt_total]
    labels = ["Futures", "Options"]
    colors = [FUT_COLOR, OPT_COLOR]
    pcts   = [fut_total / total * 100, opt_total / total * 100]

    wedges, _ = ax.pie(
        sizes,
        labels=None,            # suppress default labels — place manually
        colors=colors,
        startangle=90,
        wedgeprops=dict(width=0.52, edgecolor="white", linewidth=2.5),
        counterclock=False,
    )

    # Centre text: framework name + total
    ax.text(0, 0.12, fw_name, ha="center", va="center",
            fontsize=13, fontweight="bold", color=fw_color)
    ax.text(0, -0.18, f"₹{total/1e5:.1f}L", ha="center", va="center",
            fontsize=11, color="#333")

    # Manual legend-style labels outside — placed using annotation with no overlap
    for i, (label, pct, val, color) in enumerate(zip(labels, pcts, sizes, colors)):
        angle = 90 - (sum(pcts[:i]) + pct / 2)
        rad   = np.deg2rad(angle)
        # Text outside donut
        xtext = 1.35 * np.cos(rad)
        ytext = 1.35 * np.sin(rad)
        ax.annotate(
            f"{label}\n₹{val/1e5:.1f}L\n({pct:.1f}%)",
            xy=(0.76 * np.cos(rad), 0.76 * np.sin(rad)),
            xytext=(xtext, ytext),
            ha="center", va="center",
            fontsize=9, color="white" if color == FUT_COLOR else "#1a1a2e",
            fontweight="bold",
            arrowprops=dict(arrowstyle="-", color="#AAA", lw=0.8),
            bbox=dict(boxstyle="round,pad=0.3", fc=color, ec="white",
                      alpha=0.92, linewidth=1.2)
        )

    ax.set_title(fw_name, fontsize=13, fontweight="bold",
                 pad=18, color=fw_color)

fig.suptitle(
    "Futures vs Options Margin Split\n"
    "SPAN  |  SPAN-2  |  STANS  —  MCX Commodity Derivatives",
    fontsize=13, fontweight="bold", y=1.02, color="#1a1a2e"
)

plt.tight_layout(pad=2.5)

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig6_futuresoptionspie.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig6_futuresoptionspie.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

Saved → /content/fig6_futuresoptionspie.png


In [21]:
"""
Fig 7 : Long vs Short Position Margin Comparison
Grouped bar — Buy vs Sell, each framework side by side
Saves as fig7_longshortbar.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    port_meta[int(row[0])] = {"side": str(row[4]).strip(), "inst": str(row[3]).strip()}

ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn = int(row[0])
    rows_data.append({
        "side":  port_meta.get(rn, {}).get("side", "Buy"),
        "inst":  port_meta.get(rn, {}).get("inst", "Future"),
        "span":  float(row[5]),
        "span2": float(row[6]),
        "stans": float(row[7]),
    })

# Aggregate: side × inst type × framework
groups = {
    ("Buy",  "Future"): {"span": 0, "span2": 0, "stans": 0},
    ("Buy",  "Option"): {"span": 0, "span2": 0, "stans": 0},
    ("Sell", "Future"): {"span": 0, "span2": 0, "stans": 0},
    ("Sell", "Option"): {"span": 0, "span2": 0, "stans": 0},
}
for d in rows_data:
    key = (d["side"], d["inst"])
    if key in groups:
        for k in ("span", "span2", "stans"):
            groups[key][k] += d[k]

# ── Plot ──────────────────────────────────────────────────────────────────────
SPAN_COL  = "#2166AC"
SPAN2_COL = "#F4A261"
STANS_COL = "#2A9D8F"

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
fig.patch.set_facecolor("#F8F9FA")

titles   = ["Long (Buy) Positions", "Short (Sell) Positions"]
sides    = ["Buy", "Sell"]
inst_lbl = ["Futures", "Options"]
x        = np.arange(2)
w        = 0.22

for ax, side, title in zip(axes, sides, titles):
    ax.set_facecolor("white")

    for offset, (key, color, fw) in enumerate(
        zip(("span", "span2", "stans"),
            (SPAN_COL, SPAN2_COL, STANS_COL),
            ("SPAN", "SPAN-2", "STANS"))
    ):
        vals = [
            groups[(side, "Future")][key] / 1e5,
            groups[(side, "Option")][key] / 1e5,
        ]
        xpos = x + (offset - 1) * w
        bars = ax.bar(xpos, vals, w, color=color, label=fw,
                      edgecolor="white", linewidth=1.0, zorder=3)
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.15,
                        f"₹{val:.1f}L",
                        ha="center", va="bottom", fontsize=8, color="#1a1a2e")

    ax.set_xticks(x)
    ax.set_xticklabels(inst_lbl, fontsize=12)
    ax.set_ylabel("Margin (₹ Lakhs)", fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.0f}L"))
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10, color="#1a1a2e")
    ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylim(0, max(
        groups[(side, "Future")]["span"],
        groups[(side, "Option")]["span"]
    ) / 1e5 * 1.30)

    if ax == axes[0]:
        ax.legend(fontsize=10, framealpha=0.9, loc="upper right")

fig.suptitle(
    "Long vs Short Position Margin\n"
    "SPAN  |  SPAN-2  |  STANS  —  Futures & Options breakdown",
    fontsize=13, fontweight="bold", y=1.02, color="#1a1a2e"
)

plt.tight_layout(pad=2.0)

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig7_longshortbar.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig7_longshortbar.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

Saved → /content/fig7_longshortbar.png


In [22]:
"""
Fig 8 : Contract Value vs Margin Requirement — Scatter
3 panels (SPAN / SPAN-2 / STANS), colour = CC group, shape = Futures/Option
Saves as fig8_scattercontractmargin.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_rows = []
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    rn   = int(row[0])
    inst = str(row[3]).strip()
    side = str(row[4]).strip()
    qty  = int(row[5])
    lot  = float(row[12])
    if inst == "Future":
        cv = float(row[7]) * lot * 100 * qty
    else:
        cv = float(row[8]) * lot * 100 * qty if row[8] else 0
    port_rows.append({
        "row":  rn,
        "cv":   cv,
        "cc":   str(row[2]).strip(),
        "inst": inst,
        "side": side,
        "commodity": str(row[1]).strip(),
    })

ws_cmp = wb["Framework_Comparison"]
margins = {}
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    margins[int(row[0])] = {
        "span": float(row[5]), "span2": float(row[6]), "stans": float(row[7])
    }

# Merge
data = []
for p in port_rows:
    m = margins.get(p["row"])
    if m:
        data.append({**p, **m})

# ── Style ──────────────────────────────────────────────────────────────────────
CC_COLORS = {
    "Spices": "#E63946",
    "Guar":   "#457B9D",
    "Castor": "#2A9D8F",
    "Cotton": "#E9C46A",
}
FRAMEWORKS = [
    ("SPAN",   "span",  "#2166AC"),
    ("SPAN-2", "span2", "#F4A261"),
    ("STANS",  "stans", "#2A9D8F"),
]

ABBREV = {
    "Turmeric": "Turm", "Coriander": "Cori", "Jeera": "Jeer",
    "Guar Gum": "GGum", "Guar Seed": "GSed", "Castor Seed": "Cast",
    "Cotton Seed OC": "CSOC", "Kapas (Cotton)": "Kaps", "Kapas": "Kaps",
}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor("#F8F9FA")

for ax, (fw_name, key, fw_color) in zip(axes, FRAMEWORKS):
    ax.set_facecolor("white")

    for d in data:
        color  = CC_COLORS.get(d["cc"], "#888")
        marker = "o" if d["inst"] == "Future" else "^"
        size   = 90  if d["inst"] == "Future" else 80
        edge   = "white"
        ax.scatter(
            d["cv"] / 1e5,
            d[key] / 1e4,
            c=color, marker=marker, s=size,
            edgecolors=edge, linewidths=0.8,
            alpha=0.88, zorder=3
        )
        # Row label — offset to avoid overlap with marker
        ax.annotate(
            f"R{d['row']}",
            xy=(d["cv"] / 1e5, d[key] / 1e4),
            xytext=(4, 4), textcoords="offset points",
            fontsize=6.5, color="#444", zorder=4
        )

    # Trend line
    cv_arr = np.array([d["cv"] / 1e5 for d in data])
    mg_arr = np.array([d[key] / 1e4 for d in data])
    if len(cv_arr) > 1:
        m, b   = np.polyfit(cv_arr, mg_arr, 1)
        x_line = np.linspace(cv_arr.min(), cv_arr.max(), 100)
        ax.plot(x_line, m * x_line + b, color=fw_color,
                linestyle="--", linewidth=1.4, alpha=0.7, label="Trend")

    ax.set_xlabel("Contract Value (₹ Lakhs)", fontsize=9.5)
    ax.set_ylabel("Margin (₹ '0000)" if ax == axes[0] else "", fontsize=9.5)
    ax.set_title(fw_name, fontsize=12, fontweight="bold", pad=10, color=fw_color)
    ax.grid(linestyle="--", alpha=0.3, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Shared legend — CC group colours + instrument markers
cc_handles = [mpatches.Patch(color=v, label=k) for k, v in CC_COLORS.items()]
fut_handle = mlines.Line2D([], [], color="#555", marker="o", linestyle="None",
                           markersize=8, label="Future")
opt_handle = mlines.Line2D([], [], color="#555", marker="^", linestyle="None",
                           markersize=8, label="Option")
fig.legend(
    handles=cc_handles + [fut_handle, opt_handle],
    loc="lower center", ncol=6, fontsize=9.5,
    bbox_to_anchor=(0.5, -0.04), framealpha=0.9
)

fig.suptitle(
    "Contract Value vs Margin Requirement\n"
    "SPAN  |  SPAN-2  |  STANS  —  Colour = CC Group  |  Shape = Instrument",
    fontsize=13, fontweight="bold", y=1.02, color="#1a1a2e"
)

plt.tight_layout(rect=[0, 0.06, 1, 1])

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig8_scattercontractmargin.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig8_scattercontractmargin.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

Saved → /content/fig8_scattercontractmargin.png


In [25]:
"""
Fig 9 : Framework Capability Radar Chart
SPAN vs SPAN-2 vs STANS
Saves as fig9_radarchart_clear.png
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Scores out of 10
categories = [
    "Scenario\nCoverage",
    "Vol-Surface\nAccuracy",
    "Option\nRevaluation",
    "Calendar\nSpread",
    "Cross-CC\nOffset",
    "Deep ITM\nAccuracy",
    "Deep OTM\nFloor",
    "Tail Risk\nCapture",
    "Computational\nEfficiency",
]

scores = {
    "SPAN":   [4, 3, 3, 4, 3, 3, 5, 3, 10],
    "SPAN-2": [7, 7, 6, 7, 7, 7, 7, 6, 6],
    "STANS":  [10, 9, 10, 9, 10, 10, 9, 10, 2],
}

colors = {
    "SPAN":   "#2166AC",
    "SPAN-2": "#F4A261",
    "STANS":  "#2A9D8F",
}

N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
angles_closed = np.append(angles, angles[0])

# Plot
fig, ax = plt.subplots(figsize=(12, 11), subplot_kw=dict(polar=True))

fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("#FAFAFA")

# Axis settings
ax.set_theta_offset(0)
ax.set_theta_direction(1)
ax.set_ylim(0, 11.5)

# Grid numbers
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(["2", "4", "6", "8", "10"], fontsize=8, color="#777777")
ax.set_rlabel_position(22)

ax.grid(color="#CCCCCC", linestyle="--", linewidth=0.7, alpha=0.6)
ax.spines["polar"].set_color("#CCCCCC")

# Category labels
ax.set_xticks(angles)
ax.set_xticklabels(categories, fontsize=10, color="#1a1a2e", linespacing=1.4)
ax.tick_params(axis="x", pad=28)

# Plot each framework
for framework, vals in scores.items():
    vals_closed = np.append(vals, vals[0])

    ax.plot(
        angles_closed,
        vals_closed,
        color=colors[framework],
        linewidth=2.3,
        zorder=3
    )

    ax.fill(
        angles_closed,
        vals_closed,
        color=colors[framework],
        alpha=0.10,
        zorder=2
    )

    ax.scatter(
        angles,
        vals,
        color=colors[framework],
        s=45,
        edgecolors="white",
        linewidths=0.9,
        zorder=4
    )

# Score annotations
angle_shift = {
    "SPAN": -0.035,
    "SPAN-2": 0.000,
    "STANS": 0.035,
}

radial_shift = {
    "SPAN": 0.45,
    "SPAN-2": 0.60,
    "STANS": 0.75,
}

for i, angle in enumerate(angles):
    for framework, vals in scores.items():
        val = vals[i]

        # Keep labels at 10 inside the border
        if val >= 10:
            label_radius = 10.35
        else:
            label_radius = val + radial_shift[framework]

        ax.text(
            angle + angle_shift[framework],
            label_radius,
            str(val),
            ha="center",
            va="center",
            fontsize=8,
            color=colors[framework],
            fontweight="bold",
            zorder=5
        )

# Title
ax.set_title(
    "Framework Capability Radar\nSPAN vs SPAN-2 vs STANS",
    fontsize=15,
    fontweight="bold",
    pad=75,
    color="#1a1a2e"
)

# Legend
handles = [
    mpatches.Patch(color=colors[name], label=name)
    for name in scores.keys()
]

ax.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=3,
    fontsize=12,
    framealpha=0.95
)

# Spacing
plt.subplots_adjust(
    top=0.78,
    bottom=0.16,
    left=0.08,
    right=0.92
)

# Save and show
save_path = "fig9_radarchart_clear.png"
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved →", os.path.abspath(save_path))

Saved → /content/fig9_radarchart_clear.png


In [26]:
"""
Fig 10 : Executive Summary Dashboard
KPI cards + mini position bar + framework summary table
Saves as fig10_dashboard.png
"""

import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "openpyxl", "-q"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Load data ─────────────────────────────────────────────────────────────────
_FILENAME = "Common_Portfolio_SPAN_SPAN2_STANS.xlsx"
_SEARCH   = ["/content", os.getcwd(), "/sessions/busy-intelligent-johnson/mnt/uploads"]
try:    _SEARCH.append(os.path.dirname(os.path.abspath(__file__)))
except: pass
xlsx_path = next((os.path.join(d, _FILENAME)
                  for d in _SEARCH if os.path.exists(os.path.join(d, _FILENAME))), None)
if not xlsx_path: sys.exit(f"ERROR: {_FILENAME} not found.")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_port = wb["Common_Portfolio"]
port_meta = {}
for row in ws_port.iter_rows(min_row=3, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    port_meta[int(row[0])] = {
        "cc":   str(row[2]).strip(),
        "inst": str(row[3]).strip(),
        "side": str(row[4]).strip(),
    }

ws_cmp = wb["Framework_Comparison"]
rows_data = []
for row in ws_cmp.iter_rows(min_row=4, values_only=True):
    if row[0] is None or not isinstance(row[0], (int, float)): break
    if row[5] is None: break
    rn = int(row[0])
    rows_data.append({
        **port_meta.get(rn, {}),
        "row":   rn,
        "span":  float(row[5]),
        "span2": float(row[6]),
        "stans": float(row[7]),
    })

n       = len(rows_data)
span_m  = np.array([d["span"]  for d in rows_data])
span2_m = np.array([d["span2"] for d in rows_data])
stans_m = np.array([d["stans"] for d in rows_data])
T_SPAN, T_SPAN2, T_STANS = span_m.sum(), span2_m.sum(), stans_m.sum()

red_s2 = (T_SPAN - T_SPAN2) / T_SPAN * 100
red_st = (T_SPAN - T_STANS) / T_SPAN * 100
n_fut  = sum(1 for d in rows_data if d.get("inst") == "Future")
n_opt  = sum(1 for d in rows_data if d.get("inst") == "Option")

SPAN_COL  = "#2166AC"
SPAN2_COL = "#F4A261"
STANS_COL = "#2A9D8F"
DARK      = "#1a1a2e"
BG        = "#1a1a2e"

# ── Layout ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor(BG)

gs = GridSpec(3, 6, figure=fig,
              hspace=0.55, wspace=0.45,
              left=0.04, right=0.96,
              top=0.88,  bottom=0.08)

# ── Title ─────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.95, "FRAMEWORK COMPARISON  —  EXECUTIVE SUMMARY",
         ha="center", fontsize=17, fontweight="bold", color="white")
fig.text(0.5, 0.915,
         "MCX Commodity Derivatives Portfolio  |  SPAN  vs  SPAN-2  vs  STANS  |  24 Positions",
         ha="center", fontsize=10, color="#AAB7C4")

# ── Helper: KPI card ──────────────────────────────────────────────────────────
def kpi_card(ax, title, value, sub, bg_color, val_color="white"):
    ax.set_facecolor(bg_color)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(0.5, 0.72, value,  transform=ax.transAxes,
            ha="center", va="center", fontsize=17, fontweight="bold", color=val_color)
    ax.text(0.5, 0.32, title,  transform=ax.transAxes,
            ha="center", va="center", fontsize=8.5, color="white", linespacing=1.4)
    if sub:
        ax.text(0.5, 0.10, sub, transform=ax.transAxes,
                ha="center", va="center", fontsize=7.5,
                color="#DDDDDD" if val_color == "white" else "#333")

# Row 1 — 6 KPI cards
kpi_data = [
    ("SPAN\nGross Margin",    f"₹{T_SPAN/1e5:.1f}L",   f"₹{T_SPAN:,.0f}",   SPAN_COL,  "white"),
    ("SPAN-2\nGross Margin",  f"₹{T_SPAN2/1e5:.1f}L",  f"₹{T_SPAN2:,.0f}",  SPAN2_COL, "white"),
    ("STANS\nGross Margin",   f"₹{T_STANS/1e5:.1f}L",  f"₹{T_STANS:,.0f}",  STANS_COL, "white"),
    ("SPAN → SPAN-2\nSaving", f"−{red_s2:.1f}%",        f"₹{(T_SPAN-T_SPAN2)/1e5:.1f}L saved", "#E9C46A", "#1a1a2e"),
    ("SPAN → STANS\nSaving",  f"−{red_st:.1f}%",        f"₹{(T_SPAN-T_STANS)/1e5:.1f}L saved", "#95D5B2", "#1a1a2e"),
    ("Portfolio\nPositions",  str(n),                   f"{n_fut} Fut  |  {n_opt} Opt", "#6C757D", "white"),
]
for col, (title, val, sub, bg, vc) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, col])
    kpi_card(ax, title, val, sub, bg, vc)

# Row 2 — mini grouped bar (all 24 positions)
ax_bar = fig.add_subplot(gs[1, :])
ax_bar.set_facecolor("#252550")
x  = np.arange(n)
w  = 0.26
ax_bar.bar(x - w,   span_m  / 1e4, w, color=SPAN_COL,  label="SPAN",   alpha=0.92)
ax_bar.bar(x,       span2_m / 1e4, w, color=SPAN2_COL, label="SPAN-2", alpha=0.92)
ax_bar.bar(x + w,   stans_m / 1e4, w, color=STANS_COL, label="STANS",  alpha=0.92)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels([f"R{d['row']}" for d in rows_data],
                        fontsize=7, color="white", rotation=0)
ax_bar.set_ylabel("Margin\n(₹ '0000)", fontsize=8, color="white")
ax_bar.tick_params(axis="y", colors="white", labelsize=7)
ax_bar.grid(axis="y", alpha=0.15, color="white", linestyle="--")
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
for sp in ["bottom", "left"]:
    ax_bar.spines[sp].set_color("#555")
ax_bar.set_title("Position-Level Margin  —  All 24 Positions",
                 fontsize=10, color="white", pad=6)
ax_bar.legend(fontsize=8.5, facecolor="#252550",
              labelcolor="white", framealpha=0.85,
              loc="upper right", ncol=3)

# Row 3 — summary table
ax_tbl = fig.add_subplot(gs[2, :])
ax_tbl.set_facecolor(BG)
ax_tbl.axis("off")

table_data = [
    ["Scenario Array",        "16-scenario grid",           "50-scenario (10×5 vol)",       "10,000 MC + 500 HS paths"],
    ["Options Pricing",       "Black-Scholes flat vol",     "Vol-surface skew + term",       "Full revaluation per path"],
    ["Calendar Spread",       "Fixed ICSC table",           "Term-structure calibrated",      "Stochastic basis CVaR"],
    ["Cross-CC Credit",       "2 pairs, fixed %",           "5 pairs, corr. matrix",         "Full portfolio netting"],
    ["Deep OTM Short",        "SOM floor (0.50%×F)",        "Refined SOM (0.48%×F)",         "CVaR replaces SOM"],
    ["Gross Margin",          f"₹{T_SPAN/1e5:.2f}L",        f"₹{T_SPAN2/1e5:.2f}L",         f"₹{T_STANS/1e5:.2f}L"],
    ["vs SPAN",               "Baseline",                   f"−{red_s2:.1f}%",               f"−{red_st:.1f}%"],
]
col_labels = ["Feature", "SPAN", "SPAN-2", "STANS"]
col_widths  = [0.22, 0.24, 0.28, 0.26]
col_colors  = ["#2C2C4A", SPAN_COL, SPAN2_COL, STANS_COL]
header_txt  = ["white", "white", "white", "white"]

# Draw header
for j, (lbl, w_frac, bg, tc) in enumerate(
        zip(col_labels, col_widths, col_colors, header_txt)):
    x0 = sum(col_widths[:j])
    rect = mpatches.FancyBboxPatch((x0 + 0.003, 0.85), w_frac - 0.006, 0.10,
                                    boxstyle="round,pad=0.01",
                                    facecolor=bg, edgecolor="none",
                                    transform=ax_tbl.transAxes, clip_on=False)
    ax_tbl.add_patch(rect)
    ax_tbl.text(x0 + w_frac / 2, 0.905, lbl,
                transform=ax_tbl.transAxes,
                ha="center", va="center", fontsize=9,
                fontweight="bold", color=tc)

# Draw rows
row_h   = 0.115
row_bgs = ["#1E1E3A", "#252545"]
for i, row_vals in enumerate(table_data):
    y_top = 0.84 - i * row_h
    bg    = row_bgs[i % 2]
    for j, (val, w_frac) in enumerate(zip(row_vals, col_widths)):
        x0 = sum(col_widths[:j])
        rect = mpatches.FancyBboxPatch((x0 + 0.003, y_top - row_h + 0.01),
                                        w_frac - 0.006, row_h - 0.015,
                                        boxstyle="round,pad=0.01",
                                        facecolor=bg, edgecolor="none",
                                        transform=ax_tbl.transAxes, clip_on=False)
        ax_tbl.add_patch(rect)
        txt_color = "white" if j == 0 else (
            "#A8D8EA" if j == 1 else ("#FFD9A0" if j == 2 else "#A8E6CF"))
        ax_tbl.text(x0 + w_frac / 2, y_top - row_h / 2 + 0.005,
                    val, transform=ax_tbl.transAxes,
                    ha="center", va="center",
                    fontsize=8, color=txt_color)

plt.tight_layout(rect=[0, 0, 1, 0.91])

# ── Save + show ───────────────────────────────────────────────────────────────
save_path = "/content/fig10_dashboard.png" if os.path.isdir("/content") \
            else os.path.join(os.getcwd(), "fig10_dashboard.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
print(f"Saved → {save_path}")
plt.show()

/tmp/ipykernel_7920/3814792038.py:190: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.91])


Saved → /content/fig10_dashboard.png
